# MultiMend — Kaggle A→Z baseline reproduction (first benchmark: QuixBugs-Python)

**Scope:** official MultiMend repository + official fine-tuned checkpoints + official QuixBugs benchmark.

This notebook deliberately does **not** run all six benchmarks. It first validates the complete pipeline on QuixBugs-Python:

`buggy/correct programs → official bug-line/context extraction → local RAG → CodeT5 checkpoint ensemble → 100 candidates/checkpoint → combine/deduplicate → patch validation → CSV/JSON summary`

No OpenAI/Claude/Gemini API is used. The model and the RAG embedding model are downloaded locally from Hugging Face.


Giai đoạn 1:

In [5]:
from pathlib import Path
import shutil

repo = Path("/kaggle/working/MultiMend28")

if repo.exists():
    shutil.rmtree(repo)

print("Old repository removed.")

Old repository removed.


In [6]:
# Cell 1 — Kaggle setup
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/ThaoTo2806/MultiMend28.git"
REPO = Path("/kaggle/working/MultiMend28")

if not REPO.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO)], check=True)

os.chdir(REPO)
print("Repository:", REPO)
print("Commit:", subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip())
print("Python:", sys.version)

try:
    import torch
    print("Torch:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
except ImportError:
    print("Torch will be checked after dependency installation.")


Cloning into '/kaggle/working/MultiMend28'...


Repository: /kaggle/working/MultiMend28
Commit: 12744d8e30d25e0c6f34bee081a209ff1f04cf6a
Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Torch: 2.4.0+cu121
CUDA available: True
GPU: Tesla T4


In [7]:
# Cell 2 — Install the repository dependencies
subprocess.run(
    [
        sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir",
        "-r", "requirements.txt", "more-itertools", "pygments",
    ],
    check=True,
)
print("Dependencies installed.")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.6/177.6 kB 50.5 MB/s eta 0:00:00
Dependencies installed.


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
diffusers 0.37.1 requires huggingface-hub<2.0,>=0.34.0, but you have huggingface-hub 0.27.1 which is incompatible.
torchaudio 2.10.0+cu128 requires torch==2.10.0, but you have torch 2.4.0 which is incompatible.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2024.6.1 which is incompatible.
gradio 5.50.0 requires huggingface-hub<2.0,>=0.33.5, but you have huggingface-hub 0.27.1 which is incompatible.
gradio 5.50.0 requires pillow<12.0,>=8.0, but you have pillow 12.3.0 which is incompatible.


In [8]:
# Download QuixBugs if it is not already present
QB_DIR = REPO / "benchmarks" / "QuixBugs"

if not (QB_DIR / "python_programs").is_dir():
    QB_DIR.parent.mkdir(parents=True, exist_ok=True)

    !git clone https://github.com/jkoppel/QuixBugs.git "{QB_DIR}"

print("QuixBugs:", QB_DIR)
print("Python programs:", len(list((QB_DIR / "python_programs").glob("*.py"))))


Cloning into '/kaggle/working/MultiMend28/benchmarks/QuixBugs'...
remote: Enumerating objects: 1123, done.
remote: Counting objects: 100% (478/478), done.
remote: Compressing objects: 100% (178/178), done.
remote: Total 1123 (delta 340), reused 300 (delta 300), pack-reused 645 (from 1)
Receiving objects: 100% (1123/1123), 1.14 MiB | 32.35 MiB/s, done.
Resolving deltas: 100% (648/648), done.
QuixBugs: /kaggle/working/MultiMend28/benchmarks/QuixBugs
Python programs: 50


In [9]:
# Cell 3 — Prepare benchmark data and checkpoints

import os
import json
import shutil
from pathlib import Path

# ================================================================
# 1. Repository
# ================================================================

REPO = Path("/kaggle/working/MultiMend28")
REPO.mkdir(parents=True, exist_ok=True)
os.chdir(REPO)

print("Repository:", REPO)


# ================================================================
# 2. Prepare QuixBugs benchmark
# ================================================================

QB_DIR = REPO / "benchmarks" / "QuixBugs"

# Download QuixBugs only if it is missing.
if not (QB_DIR / "python_programs").is_dir():
    print("\nQuixBugs not found. Downloading...")

    QB_DIR.parent.mkdir(parents=True, exist_ok=True)

    result = os.system(
        f'git clone https://github.com/jkoppel/QuixBugs.git "{QB_DIR}"'
    )

    if result != 0:
        raise RuntimeError(
            "Failed to download QuixBugs. "
            "Please enable Internet in Kaggle or attach QuixBugs as a dataset."
        )

# Verify benchmark structure.
if not (QB_DIR / "python_programs").is_dir():
    raise FileNotFoundError(
        f"QuixBugs python_programs directory is missing: {QB_DIR}"
    )

if not (QB_DIR / "correct_python_programs").is_dir():
    raise FileNotFoundError(
        f"QuixBugs correct_python_programs directory is missing: {QB_DIR}"
    )

buggy_programs = sorted(
    (QB_DIR / "python_programs").glob("*.py")
)

correct_programs = sorted(
    (QB_DIR / "correct_python_programs").glob("*.py")
)

print("\nQuixBugs benchmark:", QB_DIR)
print("Buggy programs:", len(buggy_programs))
print("Correct programs:", len(correct_programs))


# ================================================================
# 3. Locate MultiMend checkpoints
# ================================================================

MODEL_DIR = REPO / "models" / "multimend-codet5-small"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# Optional manual override.
# Example:
# os.environ["MULTIMEND_CHECKPOINT_SOURCE"] = "/kaggle/input/..."
configured_source = os.environ.get("MULTIMEND_CHECKPOINT_SOURCE")


if configured_source:
    print("\nUsing configured checkpoint source:")
    print(configured_source)

    source = Path(configured_source)

    checkpoint_dirs = sorted(
        p for p in source.rglob("checkpoint-*")
        if p.is_dir() and (p / "trainer_state.json").is_file()
    )

else:
    # Automatically search all Kaggle input datasets.
    print("\nSearching /kaggle/input/datasets for checkpoints...")

    checkpoint_dirs = sorted(
        p.parent
        for p in Path("/kaggle/input/datasets").rglob("trainer_state.json")
        if p.parent.name.startswith("checkpoint-")
    )


# Remove duplicates while preserving order.
checkpoint_dirs = sorted(set(checkpoint_dirs))


if not checkpoint_dirs:
    raise FileNotFoundError(
        "No MultiMend checkpoints found.\n"
        "Expected checkpoint directories containing trainer_state.json "
        "under /kaggle/input/datasets."
    )


print(f"\nFound {len(checkpoint_dirs)} checkpoint(s):")

for checkpoint in checkpoint_dirs:
    print(" -", checkpoint)


# ================================================================
# 4. Copy checkpoints into the repository
# ================================================================

print("\nPreparing model checkpoints...")

for checkpoint in checkpoint_dirs:

    destination = MODEL_DIR / checkpoint.name

    if destination.exists():
        print(f"Already exists: {destination.name}")
        continue

    print(f"Copying: {checkpoint.name}")

    shutil.copytree(
        checkpoint,
        destination
    )


# ================================================================
# 5. Verify final model directory
# ================================================================

installed_checkpoints = sorted(
    p for p in MODEL_DIR.glob("checkpoint-*")
    if p.is_dir() and (p / "trainer_state.json").is_file()
)

if not installed_checkpoints:
    raise FileNotFoundError(
        f"No valid checkpoints were installed in {MODEL_DIR}"
    )


print("\n" + "=" * 60)
print("Cell 3 completed successfully")
print("=" * 60)

print("\nQuixBugs:")
print("  Location:", QB_DIR)
print("  Buggy programs:", len(buggy_programs))
print("  Correct programs:", len(correct_programs))

print("\nModel:")
print("  Location:", MODEL_DIR)
print("  Checkpoints:", len(installed_checkpoints))

for checkpoint in installed_checkpoints:
    print("   -", checkpoint.name)


Repository: /kaggle/working/MultiMend28

QuixBugs benchmark: /kaggle/working/MultiMend28/benchmarks/QuixBugs
Buggy programs: 50
Correct programs: 50

Searching /kaggle/input/datasets for checkpoints...

Found 10 checkpoint(s):
 - /kaggle/input/datasets/honglcl/checkpoint-116200/checkpoint-116200
 - /kaggle/input/datasets/honglcl/checkpoint174300/checkpoint-174300
 - /kaggle/input/datasets/minhhanh123/checkpoint-232400/checkpoint-232400
 - /kaggle/input/datasets/minhhanh123/checkpoint290500/checkpoint-290500
 - /kaggle/input/datasets/nguynhongbchtrm/checkpoint464800/checkpoint-464800
 - /kaggle/input/datasets/nguynhongbchtrm/checkpoint522900/checkpoint-522900
 - /kaggle/input/datasets/nguynhongbchtrm/checkpoint581008/checkpoint-581008
 - /kaggle/input/datasets/thothanh28/checkpoint58100/checkpoint-58100
 - /kaggle/input/datasets/tramtvt/checkpoint348600/checkpoint-348600
 - /kaggle/input/datasets/tramtvt/checkpoint406700/checkpoint-406700

Preparing model checkpoints...
Copying: checkpo

In [12]:
# Cell 3 — Prepare benchmark data and checkpoints
import json
import os
import shutil
from pathlib import Path

REPO = Path("/kaggle/working/MultiMend28")
os.chdir(REPO)

# ------------------------------------------------------------------
# QuixBugs
# ------------------------------------------------------------------
QB_DIR = REPO / "benchmarks" / "QuixBugs"

if not (QB_DIR / "python_programs").is_dir():
    print("QuixBugs not found. Downloading it...")

    QB_DIR.parent.mkdir(parents=True, exist_ok=True)

    result = os.system(
        f'git clone https://github.com/jkoppel/QuixBugs.git "{QB_DIR}"'
    )

    if result != 0:
        raise RuntimeError(
            "Could not download QuixBugs. "
            "Check that Kaggle Internet is enabled."
        )

if not (QB_DIR / "python_programs").is_dir():
    raise FileNotFoundError(
        f"QuixBugs python programs are missing: {QB_DIR}"
    )

if not (QB_DIR / "correct_python_programs").is_dir():
    raise FileNotFoundError(
        f"Correct QuixBugs programs are missing: {QB_DIR}"
    )

print("QuixBugs benchmark:", QB_DIR)
print(
    "Buggy programs:",
    len(list((QB_DIR / "python_programs").glob("*.py")))
)
print(
    "Correct programs:",
    len(list((QB_DIR / "correct_python_programs").glob("*.py")))
)

# ------------------------------------------------------------------
# Checkpoints
# ------------------------------------------------------------------
CHECKPOINT_SOURCE = Path(os.environ.get(
    "MULTIMEND_CHECKPOINT_SOURCE",
    "/kaggle/input/multimend-checkpoints/models/multimend-codet5-small",
))

MODEL_DIR = REPO / "models" / "multimend-codet5-small"

checkpoint_dirs = sorted(
    p for p in CHECKPOINT_SOURCE.glob("checkpoint-*")
    if (p / "trainer_state.json").is_file()
)

if checkpoint_dirs:
    MODEL_DIR.mkdir(parents=True, exist_ok=True)

    for checkpoint in checkpoint_dirs:
        destination = MODEL_DIR / checkpoint.name

        if not destination.exists():
            shutil.copytree(checkpoint, destination)

    print(
        f"Copied {len(checkpoint_dirs)} checkpoints "
        f"from {CHECKPOINT_SOURCE}"
    )

elif not list(MODEL_DIR.glob("checkpoint-*")):
    raise FileNotFoundError(
        "No checkpoints found. Attach the checkpoint Kaggle Dataset "
        "and set MULTIMEND_CHECKPOINT_SOURCE to its model directory."
    )

print("Model directory:", MODEL_DIR)
print(
    "Checkpoints:",
    [p.name for p in sorted(MODEL_DIR.glob("checkpoint-*"))]
)


QuixBugs benchmark: /kaggle/working/MultiMend28/benchmarks/QuixBugs
Buggy programs: 50
Correct programs: 50
Model directory: /kaggle/working/MultiMend28/models/multimend-codet5-small
Checkpoints: ['checkpoint-116200', 'checkpoint-174300', 'checkpoint-232400', 'checkpoint-290500', 'checkpoint-348600', 'checkpoint-406700', 'checkpoint-464800', 'checkpoint-522900', 'checkpoint-58100', 'checkpoint-581008']


In [13]:
# Cell 4 — Build QuixBugs-Python metadata WITHOUT RAG / Embedding

from pathlib import Path
import json
import difflib

# ------------------------------------------------------------------
# Paths
# ------------------------------------------------------------------

quixbugs_dir = REPO / "benchmarks" / "QuixBugs"

# QuixBugs-Python thường có các thư mục này
buggy_dir = quixbugs_dir / "python_programs"
correct_dir = quixbugs_dir / "correct_python_programs"

output_dir = REPO / "outputs" / "QuixBugs-Python"
output_dir.mkdir(parents=True, exist_ok=True)

metadata_file = output_dir / "QuixBugs_Python.jsonl"
rem_file = output_dir / "rem.txt"
add_file = output_dir / "add.txt"

print("QuixBugs:", quixbugs_dir)
print("Buggy:", buggy_dir)
print("Correct:", correct_dir)


# ------------------------------------------------------------------
# Validate benchmark
# ------------------------------------------------------------------

if not buggy_dir.exists():
    raise FileNotFoundError(
        f"Buggy programs directory not found: {buggy_dir}"
    )

if not correct_dir.exists():
    raise FileNotFoundError(
        f"Correct programs directory not found: {correct_dir}"
    )


# ------------------------------------------------------------------
# Helpers
# ------------------------------------------------------------------

def read_program(path):
    return path.read_text(encoding="utf-8").splitlines()


def unified_diff(old_lines, new_lines):
    """
    Return removed and added lines between buggy and correct versions.
    """
    diff = difflib.unified_diff(
        old_lines,
        new_lines,
        lineterm="",
    )

    removed = []
    added = []

    for line in diff:
        if line.startswith("---") or line.startswith("+++"):
            continue

        if line.startswith("-"):
            removed.append(line[1:])

        elif line.startswith("+"):
            added.append(line[1:])

    return removed, added


# ------------------------------------------------------------------
# Build metadata
# ------------------------------------------------------------------

records = []
all_removed = []
all_added = []

buggy_files = sorted(buggy_dir.glob("*.py"))

print("Buggy programs:", len(buggy_files))

for buggy_path in buggy_files:

    bugid = buggy_path.stem

    correct_path = correct_dir / buggy_path.name

    if not correct_path.exists():
        print(f"WARNING: missing correct program: {bugid}")
        continue

    buggy_lines = read_program(buggy_path)
    correct_lines = read_program(correct_path)

    removed, added = unified_diff(
        buggy_lines,
        correct_lines,
    )

    # Store global patch information
    all_removed.extend(removed)
    all_added.extend(added)

    record = {
        "bugid": bugid,
        "buggy_file": str(buggy_path),
        "correct_file": str(correct_path),
        "removed": removed,
        "added": added,
    }

    records.append(record)


# ------------------------------------------------------------------
# Write JSONL
# ------------------------------------------------------------------

with metadata_file.open("w", encoding="utf-8") as f:
    for record in records:
        f.write(
            json.dumps(
                record,
                ensure_ascii=False,
            )
            + "\n"
        )


# ------------------------------------------------------------------
# Write rem.txt / add.txt
# ------------------------------------------------------------------

rem_file.write_text(
    "\n".join(all_removed) + ("\n" if all_removed else ""),
    encoding="utf-8",
)

add_file.write_text(
    "\n".join(all_added) + ("\n" if all_added else ""),
    encoding="utf-8",
)


# ------------------------------------------------------------------
# Validate
# ------------------------------------------------------------------

required_metadata = [
    metadata_file,
    rem_file,
    add_file,
]

missing = [
    str(path)
    for path in required_metadata
    if not path.is_file()
]

if missing:
    raise FileNotFoundError(
        f"Metadata generation failed: {missing}"
    )

bug_count = sum(
    1
    for _ in metadata_file.open(encoding="utf-8")
)

print()
print("========================================")
print("Metadata generation complete")
print("========================================")
print("Metadata:", metadata_file)
print("rem.txt:", rem_file)
print("add.txt:", add_file)
print("Bug records:", bug_count)
print("Removed lines:", len(all_removed))
print("Added lines:", len(all_added))


QuixBugs: /kaggle/working/MultiMend28/benchmarks/QuixBugs
Buggy: /kaggle/working/MultiMend28/benchmarks/QuixBugs/python_programs
Correct: /kaggle/working/MultiMend28/benchmarks/QuixBugs/correct_python_programs
Buggy programs: 50

Metadata generation complete
Metadata: /kaggle/working/MultiMend28/outputs/QuixBugs-Python/QuixBugs_Python.jsonl
rem.txt: /kaggle/working/MultiMend28/outputs/QuixBugs-Python/rem.txt
add.txt: /kaggle/working/MultiMend28/outputs/QuixBugs-Python/add.txt
Bug records: 50
Removed lines: 742
Added lines: 540


In [14]:
# Cell 4 — Build official QuixBugs-Python metadata

import json
from src.bugline_finders import quixbugs_python

metadata_file = (
    REPO
    / "outputs"
    / "QuixBugs-Python"
    / "QuixBugs_Python.jsonl"
)

metadata_file.parent.mkdir(parents=True, exist_ok=True)

print("=" * 60)
print("Official QuixBugs-Python extraction")
print("=" * 60)

print("Buggy:", QB_DIR / "python_programs")
print("Correct:", QB_DIR / "correct_python_programs")
print("Official module:", quixbugs_python.__file__)
print()

# ------------------------------------------------------------
# Run the official MultiMend extractor.
#
# IMPORTANT:
# The official extractor expects every input program to have
# a diff. Some QuixBugs *_test.py files and `node.py` are
# identical between buggy/correct directories, so we filter
# those no-diff programs before calling process_hunks().
# ------------------------------------------------------------

buggy_dir = QB_DIR / "python_programs"
correct_dir = QB_DIR / "correct_python_programs"

buggy_programs = sorted(buggy_dir.glob("*.py"))

print("Buggy programs:", len(buggy_programs))
print()
print("Running official extraction...")
print()

# We temporarily use the official helper functions directly
# for each program. This preserves the official diff/hunk logic
# without forcing the official `main()` to crash on no-diff files.

programs_hunks = {}

skipped = []
failed = []

for buggy_file in buggy_programs:
    program = buggy_file.stem
    correct_file = correct_dir / buggy_file.name

    if not correct_file.is_file():
        failed.append(
            (program, f"Missing correct file: {correct_file}")
        )
        continue

    # Official MultiMend diff extraction
    diff_lines = list(
        quixbugs_python.get_diff_lines(
            buggy_file,
            correct_file,
        )
    )

    # No diff = not an actual buggy/correct pair.
    if not diff_lines:
        skipped.append(program)

        print(
            f"SKIP: {program:<35} no diff"
        )
        continue

    try:
        # Official MultiMend hunk extraction
        hunks = quixbugs_python.process_hunks(diff_lines)

        if len(hunks) != 1:
            raise RuntimeError(
                f"Expected exactly 1 hunk, got {len(hunks)}"
            )

        # Official context extraction
        for hunk in hunks:
            line_number = hunk.removed_line_numbers_range[0]
            lines_range = hunk.removed_line_numbers_range[1]

            hunk.source_context = quixbugs_python.get_context(
                buggy_file,
                line_number,
                lines_range,
            )

            quixbugs_python.remove_context_from_source(
                hunk,
                buggy_file,
            )

        programs_hunks[program] = hunks

        print(
            f"OK:   {program:<35} "
            f"diff={len(diff_lines):>3}, hunks={len(hunks)}"
        )

    except Exception as e:
        failed.append((program, repr(e)))

        print(
            f"FAILED: {program:<32} {repr(e)}"
        )


# ------------------------------------------------------------
# Fail only on genuine extraction errors
# ------------------------------------------------------------

if failed:
    print()
    print("=" * 60)
    print("EXTRACTION FAILURES")
    print("=" * 60)

    for program, error in failed:
        print(f"{program}: {error}")

    raise RuntimeError(
        f"Official extraction failed for {len(failed)} program(s)."
    )


# ------------------------------------------------------------
# Generate the official MultiMend metadata files.
#
# generate_data() is the same function used by the official
# quixbugs_python.main().
# ------------------------------------------------------------

quixbugs_python.generate_data(programs_hunks)


# ------------------------------------------------------------
# Verify generated files
# ------------------------------------------------------------

required_metadata = [
    metadata_file,
    metadata_file.parent / "rem.txt",
    metadata_file.parent / "add.txt",
]

missing = [
    str(path)
    for path in required_metadata
    if not path.is_file()
]

if missing:
    raise FileNotFoundError(
        f"Metadata generation failed: {missing}"
    )


# ------------------------------------------------------------
# Count metadata records
# ------------------------------------------------------------

metadata_records = []

with metadata_file.open(encoding="utf-8") as f:
    for line in f:
        line = line.strip()

        if line:
            metadata_records.append(json.loads(line))

bug_count = len(metadata_records)


# ------------------------------------------------------------
# Final report
# ------------------------------------------------------------

print()
print("=" * 60)
print("CELL 4 COMPLETED")
print("=" * 60)

print(f"Total benchmark files: {len(buggy_programs)}")
print(f"Extracted programs:    {len(programs_hunks)}")
print(f"Skipped no-diff:       {len(skipped)}")
print(f"Extraction failures:   {len(failed)}")

if skipped:
    print()
    print("Skipped programs:")

    for program in skipped:
        print(f"  - {program}")

print()
print("Generated files:")
print("  JSONL:  ", metadata_file)
print("  rem.txt:", metadata_file.parent / "rem.txt")
print("  add.txt:", metadata_file.parent / "add.txt")

print()
print("Metadata records:", bug_count)

# ------------------------------------------------------------
# Verify the known bitcount example
# ------------------------------------------------------------

if "bitcount" in programs_hunks:
    bitcount_hunk = programs_hunks["bitcount"][0]

    print()
    print("bitcount verification:")
    print(
        "  removed:",
        repr(bitcount_hunk.removed_lines),
    )
    print(
        "  added:  ",
        repr(bitcount_hunk.added_lines),
    )
    print(
        "  range:  ",
        bitcount_hunk.removed_line_numbers_range,
    )


Official QuixBugs-Python extraction
Buggy: /kaggle/working/MultiMend28/benchmarks/QuixBugs/python_programs
Correct: /kaggle/working/MultiMend28/benchmarks/QuixBugs/correct_python_programs
Official module: /kaggle/working/MultiMend28/src/bugline_finders/quixbugs_python.py

Buggy programs: 50

Running official extraction...

OK:   bitcount                            diff= 25, hunks=1
OK:   breadth_first_search                diff= 60, hunks=1
SKIP: breadth_first_search_test           no diff
OK:   bucketsort                          diff= 12, hunks=1
OK:   depth_first_search                  diff= 21, hunks=1
SKIP: depth_first_search_test             no diff
OK:   detect_cycle                        diff=  7, hunks=1
OK:   detect_cycle_test                   diff= 12, hunks=1
OK:   find_first_in_sorted                diff= 29, hunks=1
OK:   find_in_sorted                      diff= 26, hunks=1
OK:   flatten                             diff= 29, hunks=1
OK:   gcd                          

In [22]:
# Cell 5 - Generate both experimental conditions
import os
import subprocess
import sys

output_root = REPO / "outputs" / "QuixBugs-Python"
generator_path = REPO / "src" / "generate_primary_candidates.py"
if not generator_path.is_file():
    raise FileNotFoundError(f"Missing generator source: {generator_path}")

source = generator_path.read_text(encoding="utf-8")
required_markers = (
    'context_strategy = os.environ.get("MULTIMEND_CONTEXT_STRATEGY", "fixed_rag")',
    "decoder_start_token_id = next(",
)
missing_markers = [marker for marker in required_markers if marker not in source]
if missing_markers:
    raise RuntimeError(
        "The checked-out Python source is incomplete. "
        f"Missing markers: {missing_markers}. "
        "Push the updated src files and clone that revision on Kaggle."
    )

def run_generation(strategy: str) -> None:
    strategy_dir = output_root / f"outputs-multimend-{strategy}"
    raw_file = strategy_dir / "sequences_100.jsonl"
    if raw_file.is_file() and raw_file.stat().st_size > 0:
        print(f"{strategy}: existing output found, skipping generation")
        return
    env = os.environ.copy()
    env["MULTIMEND_CONTEXT_STRATEGY"] = strategy
    env["PYTHONPATH"] = str(REPO)
    print(f"Starting {strategy}...")
    subprocess.run(
        [sys.executable, "-m", "src.generate_primary_candidates"],
        cwd=REPO,
        env=env,
        check=True,
    )
    if not raw_file.is_file():
        raise RuntimeError(f"{strategy}: generator did not create {raw_file}")
    print(f"{strategy}: generation complete ({raw_file.stat().st_size:,} bytes)")

for strategy in ("no_rag", "fixed_rag"):
    run_generation(strategy)



no_rag: existing output found, skipping generation
Starting fixed_rag...


/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


Selected checkpoints:
- checkpoint-348600: global_step=348600
- checkpoint-406700: global_step=406700
- checkpoint-464800: global_step=464800
- checkpoint-522900: global_step=522900
- checkpoint-581008: global_step=581008
# RAG retrievals...
bitcount 0
breadth_first_search 0
bucketsort 0
depth_first_search 0
detect_cycle 0
find_first_in_sorted 0
find_in_sorted 0
flatten 0
gcd 0
get_factors 0
hanoi 0
is_valid_parenthesization 0
kheapsort 0
knapsack 0
kth 0
lcs_length 0
levenshtein 0
lis 0
longest_common_subsequence 0
max_sublist_sum 0
mergesort 0
minimum_spanning_tree 0
next_palindrome 0
next_permutation 0
pascal 0
possible_change 0
powerset 0
quicksort 0
reverse_linked_list 0
rpn_eval 0
shortest_path_length 0
shortest_path_lengths 0
shortest_paths 0
shunting_yard 0
sieve 0
sqrt 0
subsequences 0
to_base 0
topological_ordering 0
wrap 0
Tokenizing...


Map: 100%|██████████| 40/40 [00:00<00:00, 3012.34 examples/s]


Generating from checkpoint-348600...


...

Generating from checkpoint-522900...


Map: 100%|██████████| 40/40 [00:24<00:00,  1.61 examples/s]


Generating from checkpoint-581008...


Creating json from Arrow format: 100%|██████████| 20/20 [00:00<00:00, 373.58ba/s]


fixed_rag: generation complete (3,117,214 bytes)


In [20]:
import subprocess
import sys
import os

env = os.environ.copy()
env["PYTHONPATH"] = str(REPO)

result = subprocess.run(
    [sys.executable, "-m", "src.bugline_finders.quixbugs_python"],
    cwd=REPO,
    env=env,
    text=True,
)

print("Return code:", result.returncode)


/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event CollectionAddEvent: capture() takes 1 positional argument but 3 were given


Embedding...
bitcount 0
breadth_first_search 0
bucketsort 0
depth_first_search 0
detect_cycle 0
find_first_in_sorted 0
find_in_sorted 0
flatten 0
gcd 0
get_factors 0
hanoi 0
is_valid_parenthesization 0
kheapsort 0
knapsack 0
kth 0
lcs_length 0
levenshtein 0
lis 0
longest_common_subsequence 0
max_sublist_sum 0
mergesort 0
minimum_spanning_tree 0
next_palindrome 0
next_permutation 0
pascal 0
possible_change 0
powerset 0
quicksort 0
reverse_linked_list 0
rpn_eval 0
shortest_path_length 0
shortest_path_lengths 0
shortest_paths 0
shunting_yard 0
sieve 0
sqrt 0
subsequences 0
to_base 0
topological_ordering 0
wrap 0
Return code: 0


In [21]:
from src.configs import chromadb_path
import chromadb

print("ChromaDB path:", chromadb_path)

client = chromadb.PersistentClient(path=str(chromadb_path))

print("Collections hiện có:")
for collection in client.list_collections():
    print("-", collection.name)


Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given


ChromaDB path: /kaggle/working/MultiMend28/chroma_db
Collections hiện có:
- QuixBugs-Python


In [59]:
for strategy in ("no_rag", "fixed_rag"):
    run_combine(strategy)


Starting combine for no_rag...
All: 20000
Deduped: 9709
no_rag: combine complete (4,286,508 bytes)
Starting combine for fixed_rag...
All: 20000
Deduped: 9667
fixed_rag: combine complete (4,270,513 bytes)


In [60]:
import json
from collections import Counter

base = REPO / "outputs" / "QuixBugs-Python"

for strategy in ("no_rag", "fixed_rag"):
    path = (
        base
        / f"outputs-multimend-{strategy}"
        / "final_candidates_100.jsonl"
    )

    with open(path, encoding="utf-8") as f:
        rows = [json.loads(line) for line in f]

    print(f"\n{strategy}")
    print("Total final rows:", len(rows))
    print("First row keys:", rows[0].keys())

    counts = Counter(
        (row["bugid"], row["hunk"])
        for row in rows
    )

    print("Bug/hunk pairs:", len(counts))
    print("Min candidates per pair:", min(counts.values()))
    print("Max candidates per pair:", max(counts.values()))

    bad = {
        pair: n
        for pair, n in counts.items()
        if n < 100
    }

    print("Pairs with <100:", len(bad))



no_rag
Total final rows: 9709
First row keys: dict_keys(['bugid', 'hunk', 'checkpoint', 'decoded_sequences', 'sequences_scores', 'source', 'target', 'normalized_patch', 'normalized_source', 'normalized_target', 'rank', 'exact_match', 'correct'])
Bug/hunk pairs: 40
Min candidates per pair: 175
Max candidates per pair: 294
Pairs with <100: 0

fixed_rag
Total final rows: 9667
First row keys: dict_keys(['bugid', 'hunk', 'checkpoint', 'decoded_sequences', 'sequences_scores', 'source', 'target', 'normalized_patch', 'normalized_source', 'normalized_target', 'rank', 'exact_match', 'correct'])
Bug/hunk pairs: 40
Min candidates per pair: 170
Max candidates per pair: 294
Pairs with <100: 0


In [61]:
import json
from collections import Counter

base = REPO / "outputs" / "QuixBugs-Python"

for strategy in ("no_rag", "fixed_rag"):
    path = (
        base
        / f"outputs-multimend-{strategy}"
        / "final_candidates_100.jsonl"
    )

    with open(path, encoding="utf-8") as f:
        rows = [json.loads(line) for line in f]

    print(f"\n=== {strategy} ===")
    print("Rows:", len(rows))
    print("Exact matches:", sum(r["exact_match"] for r in rows))
    print("Correct:", sum(r["correct"] for r in rows))

    by_bug = Counter()
    for r in rows:
        if r["correct"]:
            by_bug[r["bugid"]] += 1

    print("Bugs with >=1 correct candidate:", len(by_bug))



=== no_rag ===
Rows: 9709
Exact matches: 27
Correct: 0
Bugs with >=1 correct candidate: 0

=== fixed_rag ===
Rows: 9667
Exact matches: 27
Correct: 0
Bugs with >=1 correct candidate: 0


In [66]:
# Cell 6 — Combine and deduplicate candidates for B0/B1

import os
import subprocess
import sys

STRATEGIES = ("no_rag", "fixed_rag")


def run_combine(strategy):
    output_dir = (
        REPO
        / "outputs"
        / "QuixBugs-Python"
        / f"outputs-multimend-{strategy}"
    )
    final_file = output_dir / "final_candidates_100.jsonl"

    output_dir.mkdir(parents=True, exist_ok=True)

    # Remove old combined output so we always combine
    # from the current generation results.
    if final_file.exists():
        final_file.unlink()
        print(f"{strategy}: removed old {final_file.name}")

    env = os.environ.copy()
    env["MULTIMEND_CONTEXT_STRATEGY"] = strategy

    print(f"\nStarting combine for {strategy}...")

    subprocess.run(
        [sys.executable, "-m", "src.combine_checkpoints_results"],
        cwd=REPO,
        env=env,
        check=True,
    )

    if not final_file.is_file() or final_file.stat().st_size == 0:
        raise RuntimeError(
            f"{strategy}: combine did not create a valid output: {final_file}"
        )

    print(
        f"{strategy}: combine complete "
        f"({final_file.stat().st_size:,} bytes)"
    )


for strategy in STRATEGIES:
    run_combine(strategy)


no_rag: removed old final_candidates_100.jsonl

Starting combine for no_rag...
All: 20000
Deduped: 9709
no_rag: combine complete (4,286,508 bytes)
fixed_rag: removed old final_candidates_100.jsonl

Starting combine for fixed_rag...
All: 20000
Deduped: 9667
fixed_rag: combine complete (4,270,513 bytes)


In [74]:
import json
import pandas as pd
from collections import ChainMap

output_size = 100

for strategy in ("no_rag", "fixed_rag"):
    output_dir = (
        REPO / "outputs" / "QuixBugs-Python"
        / f"outputs-multimend-{strategy}"
    )

    candidate_patches_df = pd.read_json(
        output_dir / f"final_candidates_{output_size}.jsonl",
        orient="records",
        lines=True,
    )

    with open(
        REPO / "outputs" / "QuixBugs-Python" / "QuixBugs_Python.jsonl"
    ) as meta_file:
        all_metadata = ChainMap(
            *[json.loads(line) for line in meta_file][::-1]
        )

    candidate_bugids = set(candidate_patches_df["bugid"].unique())

    bugs_metadata = {
        bugid: hunks
        for bugid, hunks in all_metadata.items()
        if bugid in candidate_bugids
    }

    print(f"\n=== {strategy} ===")
    print("Metadata programs:", len(all_metadata))
    print("Candidate programs:", len(candidate_bugids))
    print("Metadata used for validation:", len(bugs_metadata))
    print(
        "Skipped:",
        sorted(set(all_metadata) - candidate_bugids)
    )



=== no_rag ===
Metadata programs: 42
Candidate programs: 40
Metadata used for validation: 40
Skipped: ['detect_cycle_test', 'shortest_path_length_test']

=== fixed_rag ===
Metadata programs: 42
Candidate programs: 40
Metadata used for validation: 40
Skipped: ['detect_cycle_test', 'shortest_path_length_test']


In [101]:
import subprocess
from pathlib import Path

REPO = Path("/kaggle/working/MultiMend28")
FILE = "src/validators/validate_quixbugs_python.py"

# Xem stash hiện tại
subprocess.run(
    ["git", "stash", "list"],
    cwd=REPO,
    check=True,
)

# Lấy đúng file validator từ stash
subprocess.run(
    ["git", "checkout", "stash@{0}", "--", FILE],
    cwd=REPO,
    check=True,
)

# Đánh dấu conflict đã được resolve
subprocess.run(
    ["git", "add", FILE],
    cwd=REPO,
    check=True,
)

print("Validator local version restored from stash.")

subprocess.run(
    ["git", "status", "--short"],
    cwd=REPO,
    check=True,
)


stash@{0}: On master: validator-local-change
Validator local version restored from stash.
M  src/validators/validate_quixbugs_python.py
?? chroma_db/


CompletedProcess(args=['git', 'status', '--short'], returncode=0)

In [104]:
path = REPO / "src" / "validators" / "validate_quixbugs_python.py"

text = path.read_text(encoding="utf-8")

old = '''# Default timeout remains 60 seconds.
DEFAULT_TEST_TIMEOUT = float(os.environ.get("MULTIMEND_TEST_TIMEOUT", "60"))
CANDIDATE_SLEEP = float(os.environ.get("MULTIMEND_CANDIDATE_SLEEP", "1.0"))
'''

new = '''# Validation configuration.
# Default timeout remains 60 seconds, but can be overridden by environment.
DEFAULT_TEST_TIMEOUT = float(
    os.environ.get("MULTIMEND_VALIDATION_TIMEOUT", "60")
)

CANDIDATE_SLEEP = float(
    os.environ.get("MULTIMEND_VALIDATION_SLEEP", "1.0")
)

N_WORKERS = int(
    os.environ.get("MULTIMEND_VALIDATION_JOBS", "6")
)
'''

if old not in text:
    raise RuntimeError(
        "Không tìm thấy đúng đoạn config cũ để thay thế."
    )

text = text.replace(old, new, 1)

path.write_text(text, encoding="utf-8")

print("Đã sửa config.")
print()
print(path.read_text(encoding="utf-8").splitlines()[35:55])


Đã sửa config.

['save_state_dir = output_dir / "save-state"', 'output_size = 100', '', '# Configurable through environment variables.', '# Validation configuration.', '# Default timeout remains 60 seconds, but can be overridden by environment.', 'DEFAULT_TEST_TIMEOUT = float(', '    os.environ.get("MULTIMEND_VALIDATION_TIMEOUT", "60")', ')', '', 'CANDIDATE_SLEEP = float(', '    os.environ.get("MULTIMEND_VALIDATION_SLEEP", "1.0")', ')', '', 'N_WORKERS = int(', '    os.environ.get("MULTIMEND_VALIDATION_JOBS", "6")', ')', 'N_WORKERS = int(os.environ.get("MULTIMEND_N_WORKERS", "6"))', '', 'rem_file_path = gen_dir / "rem.txt"']


In [110]:
path = REPO / "src" / "validators" / "validate_quixbugs_python.py"

for i, line in enumerate(
    path.read_text(encoding="utf-8").splitlines(), 1
):
    if (
        "MULTIMEND_VALIDATION" in line
        or "N_WORKERS" in line
        or "DEFAULT_TEST_TIMEOUT" in line
        or "CANDIDATE_SLEEP" in line
    ):
        print(f"{i:4} | {line}")


  42 | DEFAULT_TEST_TIMEOUT = float(
  43 |     os.environ.get("MULTIMEND_VALIDATION_TIMEOUT", "60")
  46 | CANDIDATE_SLEEP = float(
  47 |     os.environ.get("MULTIMEND_VALIDATION_SLEEP", "1.0")
  50 | N_WORKERS = int(
  51 |     os.environ.get("MULTIMEND_VALIDATION_JOBS", "6")
 117 |     timeout = DEFAULT_TEST_TIMEOUT
 318 |             time.sleep(CANDIDATE_SLEEP)
 343 |     n_jobs = N_WORKERS
 347 |         f"test_timeout={DEFAULT_TEST_TIMEOUT}s, "
 348 |         f"candidate_sleep={CANDIDATE_SLEEP}s, "


In [122]:
# Cell 7A — Clean stale validator state

import shutil
from pathlib import Path

QUIXBUGS_OUTPUT = REPO / "outputs" / "QuixBugs-Python"

for strategy in ("no_rag", "fixed_rag"):
    output_dir = (
        QUIXBUGS_OUTPUT
        / f"outputs-multimend-{strategy}"
    )

    save_state_dir = output_dir / "save-state"
    result_file = output_dir / "plausible_candidates_100.jsonl"

    if save_state_dir.exists():
        shutil.rmtree(save_state_dir)
        print(f"Removed save-state: {save_state_dir}")

    # Xóa kết quả cũ để validator chạy lại
    if result_file.exists():
        result_file.unlink()
        print(f"Removed result: {result_file}")

print("Old validator state cleaned.")


Removed save-state: /kaggle/working/MultiMend28/outputs/QuixBugs-Python/outputs-multimend-no_rag/save-state
Old validator state cleaned.


In [148]:
from pathlib import Path
import shutil

src = Path("/kaggle/working/MultiMend28/src/validators/validate_quixbugs_python.py")
dst = Path("/kaggle/working/validate_quixbugs_python.py")

shutil.copy2(src, dst)

print(dst)
print(f"{dst.stat().st_size:,} bytes")


/kaggle/working/validate_quixbugs_python.py
14,437 bytes


In [125]:
path = REPO / "src" / "validators" / "validate_quixbugs_python.py"

text = path.read_text(encoding="utf-8")

marker = '''    with open(gen_dir / bugs_metadata_file) as meta_file:
'''

if "skipped_programs = {" in text:
    print("Skip logic already exists.")
else:
    pos = text.find(marker)

    if pos == -1:
        raise RuntimeError(
            "Không tìm thấy đoạn `with open(gen_dir / bugs_metadata_file)`."
        )

    # Tìm cuối block đọc metadata
    insert_marker = '''    candidate_patches_df = pd.read_json(
'''

    insert_pos = text.find(insert_marker, pos)

    if insert_pos == -1:
        raise RuntimeError(
            "Không tìm thấy candidate_patches_df."
        )

    skip_code = '''    # Programs excluded from validation.
    skipped_programs = {
        "detect_cycle_test",
        "shortest_path_length_test",
    }

    metadata_programs = len(bugs_metadata)

    bugs_metadata = {
        bugid: hunks
        for bugid, hunks in bugs_metadata.items()
        if bugid not in skipped_programs
    }

    print(f"Metadata programs: {metadata_programs}")
    print(f"Programs to validate: {len(bugs_metadata)}")
    print(f"Skipped: {sorted(skipped_programs)}")

'''

    text = text[:insert_pos] + skip_code + text[insert_pos:]

    path.write_text(text, encoding="utf-8")

    print("Đã thêm skip logic.")


Đã thêm skip logic.


In [126]:
path = REPO / "src" / "validators" / "validate_quixbugs_python.py"

lines = path.read_text(encoding="utf-8").splitlines()

for i in range(400, 445):
    if i < len(lines):
        print(f"{i+1:4} | {lines[i]}")


 401 |             n_jobs=n_jobs,
 402 |             backend="threading",
 403 |         )(
 404 |             delayed(apply_patch)(
 405 |                 deepcopy(
 406 |                     get_candidates(
 407 |                         candidate_patches_df,
 408 |                         bugid,
 409 |                     )
 410 |                 ),
 411 |                 bugid,
 412 |                 hunks,
 413 |             )
 414 |             for bugid, hunks in bugs_metadata.items()
 415 |         )
 416 | 
 417 |     cp_dfs = []
 418 | 
 419 |     for cp in sorted(save_state_dir.iterdir()):
 420 | 
 421 |         if cp.suffix != ".jsonl":
 422 |             continue
 423 | 
 424 |         df = pd.read_json(
 425 |             cp,
 426 |             orient="records",
 427 |             lines=True,
 428 |         )
 429 | 
 430 |         # -----------------------------------------------------
 431 |         # Backward compatibility:
 432 |         # old save-state may not have 

In [129]:
path = REPO / "src" / "validators" / "validate_quixbugs_python.py"

text = path.read_text(encoding="utf-8")

old = '''            bugids = df["bugid"].unique()
'''

new = '''            # Empty or malformed old save-state must not crash validation.
            if df.empty or "bugid" not in df.columns:
                print(
                    f"[VALIDATOR] Ignoring empty/malformed save-state: {cp}",
                    flush=True,
                )
                continue

            bugids = df["bugid"].unique()
'''

count = text.count(old)

print("Occurrences found:", count)

if count == 0:
    raise RuntimeError(
        'Không tìm thấy dòng `bugids = df["bugid"].unique()`.'
    )

text = text.replace(old, new, 1)

path.write_text(text, encoding="utf-8")

print("Đã thêm guard cho save-state rỗng/malformed.")


Occurrences found: 1
Đã thêm guard cho save-state rỗng/malformed.


In [130]:
path = REPO / "src" / "validators" / "validate_quixbugs_python.py"

lines = path.read_text(encoding="utf-8").splitlines()

for i, line in enumerate(lines):
    if 'bugids = df["bugid"].unique()' in line:
        for j in range(max(0, i - 8), min(len(lines), i + 8)):
            print(f"{j+1:4} | {lines[j]}")


 450 |             # Empty or malformed old save-state must not crash validation.
 451 |             if df.empty or "bugid" not in df.columns:
 452 |                 print(
 453 |                     f"[VALIDATOR] Ignoring empty/malformed save-state: {cp}",
 454 |                     flush=True,
 455 |                 )
 456 |                 continue
 457 | 
 458 |             bugids = df["bugid"].unique()
 459 | 
 460 |             for bugid in bugids:
 461 |                 if bugid not in bugs_metadata:
 462 |                     continue
 463 | 
 464 |                 bug_hunks = bugs_metadata[bugid]
 465 | 


In [131]:
path = REPO / "src" / "validators" / "validate_quixbugs_python.py"

lines = path.read_text(encoding="utf-8").splitlines()

for i, line in enumerate(lines):
    if "bugs_metadata =" in line:
        print(f"\n--- around line {i+1} ---")
        for j in range(max(0, i - 8), min(len(lines), i + 25)):
            print(f"{j+1:4} | {lines[j]}")



--- around line 354 ---
 346 |         "[VALIDATOR CONFIG] "
 347 |         f"test_timeout={DEFAULT_TEST_TIMEOUT}s, "
 348 |         f"candidate_sleep={CANDIDATE_SLEEP}s, "
 349 |         f"workers={n_jobs}",
 350 |         flush=True,
 351 |     )
 352 | 
 353 |     with open(gen_dir / bugs_metadata_file) as meta_file:
 354 |         bugs_metadata = ChainMap(
 355 |             *[json.loads(line) for line in meta_file][::-1]
 356 |         )
 357 | 
 358 |     # Programs excluded from validation.
 359 |     skipped_programs = {
 360 |         "detect_cycle_test",
 361 |         "shortest_path_length_test",
 362 |     }
 363 | 
 364 |     metadata_programs = len(bugs_metadata)
 365 | 
 366 |     bugs_metadata = {
 367 |         bugid: hunks
 368 |         for bugid, hunks in bugs_metadata.items()
 369 |         if bugid not in skipped_programs
 370 |     }
 371 | 
 372 |     print(f"Metadata programs: {metadata_programs}")
 373 |     print(f"Programs to validate: {len(bugs_metadata)}"

In [132]:
for i, line in enumerate(lines):
    if "Parallel(" in line:
        print(f"\n--- around line {i+1} ---")
        for j in range(max(0, i - 12), min(len(lines), i + 25)):
            print(f"{j+1:4} | {lines[j]}")



--- around line 400 ---
 388 | 
 389 |     shutil.rmtree(
 390 |         temp_dir,
 391 |         ignore_errors=True,
 392 |     )
 393 | 
 394 |     save_state_dir.mkdir(
 395 |         parents=True,
 396 |         exist_ok=True,
 397 |     )
 398 | 
 399 |     with tqdm_joblib(tqdm(total=len(bugs_metadata))):
 400 |         Parallel(
 401 |             n_jobs=n_jobs,
 402 |             backend="threading",
 403 |         )(
 404 |             delayed(apply_patch)(
 405 |                 deepcopy(
 406 |                     get_candidates(
 407 |                         candidate_patches_df,
 408 |                         bugid,
 409 |                     )
 410 |                 ),
 411 |                 bugid,
 412 |                 hunks,
 413 |             )
 414 |             for bugid, hunks in bugs_metadata.items()
 415 |         )
 416 | 
 417 |     cp_dfs = []
 418 | 
 419 |     for cp in sorted(save_state_dir.iterdir()):
 420 | 
 421 |         if cp.suffix != ".jsonl":
 422

In [133]:
import ast

path = REPO / "src" / "validators" / "validate_quixbugs_python.py"

text = path.read_text(encoding="utf-8")

print("skip logic:", "skipped_programs" in text)
print("40/42 logic:", "Programs to validate:" in text)
print("tqdm:", "tqdm(total=len(bugs_metadata))" in text)

print("\nValidator path:")
print(path)


skip logic: True
40/42 logic: True
tqdm: True

Validator path:
/kaggle/working/MultiMend28/src/validators/validate_quixbugs_python.py


In [135]:
import shutil

for strategy in ("no_rag", "fixed_rag"):
    output_dir = (
        REPO / "outputs" / "QuixBugs-Python"
        / f"outputs-multimend-{strategy}"
    )

    save_state = output_dir / "save-state"
    result_file = output_dir / "plausible_candidates_100.jsonl"

    if save_state.exists():
        shutil.rmtree(save_state)
        print(f"Removed: {save_state}")

    if result_file.exists():
        result_file.unlink()
        print(f"Removed: {result_file}")


Removed: /kaggle/working/MultiMend28/outputs/QuixBugs-Python/outputs-multimend-no_rag/save-state


In [134]:
import subprocess
import sys

print("REPO:", REPO)
print("Python:", sys.executable)

result = subprocess.run(
    [
        sys.executable,
        "-c",
        """
import src.validators.validate_quixbugs_python as v
print("LOADED:", v.__file__)
""",
    ],
    cwd=REPO,
    capture_output=True,
    text=True,
)

print(result.stdout)
print(result.stderr)


REPO: /kaggle/working/MultiMend28
Python: /usr/bin/python3
LOADED: /kaggle/working/MultiMend28/src/validators/validate_quixbugs_python.py




In [137]:
# Cell 7 — Validate candidates with the official QuixBugs pytest validator

import json
import os
import shutil
import subprocess
import sys
from pathlib import Path


# ============================================================
# Paths
# ============================================================

REPO = Path("/kaggle/working/MultiMend28")

QUIXBUGS_OUTPUT = REPO / "outputs" / "QuixBugs-Python"

METADATA_FILE = QUIXBUGS_OUTPUT / "QuixBugs_Python.jsonl"
BACKUP_FILE = QUIXBUGS_OUTPUT / "QuixBugs_Python.jsonl.backup"


# ============================================================
# Programs to skip
# ============================================================

SKIPPED = [
    "detect_cycle_test",
    "shortest_path_length_test",
]


# ============================================================
# Metadata information
# ============================================================

with METADATA_FILE.open("r", encoding="utf-8") as handle:
    metadata_lines = [
        line.strip()
        for line in handle
        if line.strip()
    ]

metadata_programs = len(metadata_lines)
programs_to_validate = metadata_programs - len(SKIPPED)

print(f"Metadata programs: {metadata_programs}")
print(f"Programs to validate: {programs_to_validate}")
print(f"Skipped: {SKIPPED}")


# ============================================================
# Backup metadata
# ============================================================

shutil.copy2(METADATA_FILE, BACKUP_FILE)

print(f"Backed up metadata to: {BACKUP_FILE}")


# ============================================================
# Validation
# ============================================================

def run_validation(strategy):

    strategy_output_dir = (
        QUIXBUGS_OUTPUT
        / f"outputs-multimend-{strategy}"
    )

    result_file = (
        strategy_output_dir
        / "plausible_candidates_100.jsonl"
    )

    # --------------------------------------------------------
    # Check whether an existing result already contains
    # the required `correct` column.
    # --------------------------------------------------------

    if result_file.is_file() and result_file.stat().st_size > 0:

        try:
            with result_file.open(
                "r",
                encoding="utf-8",
            ) as handle:
                first_line = handle.readline()

            first_record = json.loads(first_line)

        except (json.JSONDecodeError, OSError):
            first_record = {}

        if "correct" in first_record:
            print(
                f"{strategy}: existing validation output "
                f"with `correct` found, skipping"
            )
            return

        print(
            f"{strategy}: existing output does not contain "
            f"`correct`; validation will run again"
        )

    # --------------------------------------------------------
    # Environment configuration
    # --------------------------------------------------------

    env = os.environ.copy()

    env["MULTIMEND_CONTEXT_STRATEGY"] = strategy

    # Default in validator remains 60 seconds,
    # but Cell 7 overrides it to 10 seconds.
    env["MULTIMEND_VALIDATION_TIMEOUT"] = "10"

    # Sleep between candidates.
    env["MULTIMEND_VALIDATION_SLEEP"] = "0.05"

    # Number of workers.
    env["MULTIMEND_VALIDATION_JOBS"] = "6"

    print()
    print("=" * 70)
    print(f"VALIDATING: {strategy}")
    print("=" * 70)
    print("timeout per test : 10 seconds")
    print("sleep candidates : 0.05 seconds")
    print("workers          : 6")
    print("=" * 70)

    # --------------------------------------------------------
    # Run official validator
    # --------------------------------------------------------

    subprocess.run(
        [
            sys.executable,
            "-m",
            "src.validators.validate_quixbugs_python",
        ],
        cwd=REPO,
        env=env,
        check=True,
    )

    # --------------------------------------------------------
    # Verify output
    # --------------------------------------------------------

    if not result_file.is_file():
        raise RuntimeError(
            f"{strategy}: validator did not create:\n"
            f"{result_file}"
        )

    if result_file.stat().st_size == 0:
        raise RuntimeError(
            f"{strategy}: validation output is empty:\n"
            f"{result_file}"
        )

    # Check that both required columns exist.
    with result_file.open(
        "r",
        encoding="utf-8",
    ) as handle:
        first_line = handle.readline()

    record = json.loads(first_line)

    missing_columns = {
        "plausible",
        "correct",
    } - set(record.keys())

    if missing_columns:
        raise RuntimeError(
            f"{strategy}: output is missing columns: "
            f"{sorted(missing_columns)}"
        )

    print()
    print(
        f"{strategy}: validation complete"
    )
    print(
        f"Output: {result_file}"
    )
    print(
        "Columns verified: plausible, correct"
    )


# ============================================================
# Run both validation strategies
# ============================================================

for strategy in ("no_rag", "fixed_rag"):
    run_validation(strategy)


Metadata programs: 42
Programs to validate: 40
Skipped: ['detect_cycle_test', 'shortest_path_length_test']
Backed up metadata to: /kaggle/working/MultiMend28/outputs/QuixBugs-Python/QuixBugs_Python.jsonl.backup

VALIDATING: no_rag
timeout per test : 10 seconds
sleep candidates : 0.05 seconds
workers          : 6
[VALIDATOR CONFIG] test_timeout=10.0s, candidate_sleep=0.05s, workers=6
Metadata programs: 42
Programs to validate: 40
Skipped: ['detect_cycle_test', 'shortest_path_length_test']


  0%|          | 0/40 [00:00<?, ?it/s]

[VALIDATOR] bug=depth_first_search candidates=282
[VALIDATOR] START index=705
[VALIDATOR] bug=bucketsort candidates=241
[VALIDATOR] START index=464
[VALIDATOR] bug=find_first_in_sorted candidates=240
[VALIDATOR] START index=1242
[VALIDATOR] bug=detect_cycle candidates=255
[VALIDATOR] START index=987
[VALIDATOR] bug=breadth_first_search candidates=239
[VALIDATOR] START index=225
[VALIDATOR] bug=bitcount candidates=225
[VALIDATOR] START index=0
[VALIDATOR] START index=226
[VALIDATOR] START index=706
[VALIDATOR] START index=1243
[VALIDATOR] START index=988
[VALIDATOR] START index=465
[VALIDATOR] START index=227
[VALIDATOR] START index=466
[VALIDATOR] START index=989
[VALIDATOR] START index=1244
[VALIDATOR] START index=707
[VALIDATOR] START index=228
[VALIDATOR] START index=990
[VALIDATOR] START index=467
[VALIDATOR] START index=1245
[VALIDATOR] START index=1
[VALIDATOR] START index=708
[VALIDATOR] START index=991
[VALIDATOR] START index=229
[VALIDATOR] START index=468
[VALIDATOR] START in

  2%|▎         | 1/40 [11:57<7:46:08, 717.15s/it]

[VALIDATOR] bug=find_in_sorted candidates=237
[VALIDATOR] START index=1482
[VALIDATOR] START index=930
[VALIDATOR] START index=120
[VALIDATOR] START index=1230


...

[VALIDATOR] START index=9522
[VALIDATOR] START index=9523
[VALIDATOR] START index=9524
[VALIDATOR] START index=9525
[VALIDATOR] START index=9526
[VALIDATOR] START index=9527
[VALIDATOR] START index=9528
[VALIDATOR] START index=8554
[VALIDATOR] START index=9529
[VALIDATOR] START index=9530
[VALIDATOR] START index=9531
[VALIDATOR] START index=9532
[VALIDATOR] START index=9533
[VALIDATOR] START index=9534
[VALIDATOR] START index=9535
[VALIDATOR] START index=9536
[VALIDATOR] START index=8555
[VALIDATOR] START index=9537
[VALIDATOR] START index=9538
[VALIDATOR] START index=9539
[VALIDATOR] START index=9540
[VALIDATOR] START index=9541
[VALIDATOR] START index=9542
[VALIDATOR] START index=9543
[VALIDATOR] START index=9544
[VALIDATOR] START index=9545
[VALIDATOR] START index=8556
[VALIDATOR] START index=9546
[VALIDATOR] START index=9547
[VALIDATOR] START index=9548
[VALIDATOR] START index=9549
[VALIDATOR] START index=9550
[VALIDATOR] START index=9551
[VALIDATOR] START index=9552
[VALIDATOR] ST

 98%|█████████▊| 39/40 [1:23:19<01:55, 115.92s/it]

[VALIDATOR] START index=8590
[VALIDATOR] START index=8591
[VALIDATOR] START index=8592
[VALIDATOR] START index=8593
[VALIDATOR] START index=8594
[VALIDATOR] START index=8595
[VALIDATOR] START index=8596
[VALIDATOR] START index=8597
[VALIDATOR] START index=8598
[VALIDATOR] START index=8599
[VALIDATOR] START index=8600
[VALIDATOR] START index=8601
[VALIDATOR] START index=8602
[VALIDATOR] START index=8603
[VALIDATOR] START index=8604
[VALIDATOR] START index=8605
[VALIDATOR] START index=8606
[VALIDATOR] START index=8607
[VALIDATOR] START index=8608
[VALIDATOR] START index=8609
[VALIDATOR] START index=8610
[VALIDATOR] START index=8611
[VALIDATOR] START index=8612
[VALIDATOR] START index=8613
[VALIDATOR] START index=8614
[VALIDATOR] START index=8615
[VALIDATOR] START index=8616
[VALIDATOR] START index=8617
[VALIDATOR] START index=8618
[VALIDATOR] START index=8619
[VALIDATOR] START index=8620
[VALIDATOR] START index=8621
[VALIDATOR] START index=8622
[VALIDATOR] START index=8623
[VALIDATOR] ST

100%|██████████| 40/40 [1:31:18<00:00, 136.95s/it]



===== PLAUSIBLE =====
bugid
bitcount                       True
breadth_first_search           True
bucketsort                    False
depth_first_search             True
detect_cycle                   True
find_first_in_sorted           True
find_in_sorted                 True
flatten                        True
gcd                           False
get_factors                    True
hanoi                          True
is_valid_parenthesization      True
kheapsort                      True
knapsack                       True
kth                           False
lcs_length                     True
levenshtein                    True
lis                            True
longest_common_subsequence     True
max_sublist_sum               False
mergesort                      True
minimum_spanning_tree          True
next_palindrome                True
next_permutation               True
pascal                         True
possible_change                True
powerset                      False

In [138]:
# Cell 8 â€” Produce metrics required by the Stage 1 plan
import json
import pandas as pd

def load_jsonl(path):
    return pd.read_json(path, orient="records", lines=True)

def bug_success_at_k(frame, column, k):
    eligible = frame[frame["rank"] < k]
    per_hunk = eligible.groupby(["bugid", "hunk"])[column].any()
    return per_hunk.groupby(level=0).all()

def summarize(strategy):
    base = output_root / f"outputs-multimend-{strategy}"
    candidates = load_jsonl(base / "final_candidates_100.jsonl")
    validated = load_jsonl(base / "plausible_candidates_100.jsonl")
    if "rank" not in validated:
        raise ValueError(f"{strategy}: validator output has no rank column")
    if "exact_match" not in validated:
        validated["exact_match"] = False
    validated["exact_match"] = validated["exact_match"].fillna(False)
    if "correct" not in validated:
        validated["correct"] = False
    validated["correct"] = validated["correct"].fillna(False)
    validated["plausible"] = validated["plausible"].fillna(False)
    bugs = int(validated["bugid"].nunique())
    plausible_bugs = bug_success_at_k(validated, "plausible", 10**9)
    exact_bugs = bug_success_at_k(validated, "exact_match", 10**9)
    context_path = base / f"context_log_{strategy}.jsonl"
    context = load_jsonl(context_path) if context_path.exists() else pd.DataFrame()
    context_count = len(context)
    return {
        "strategy": strategy,
        "bugs": bugs,
        "hunks": int(validated[["bugid", "hunk"]].drop_duplicates().shape[0]),
        "raw_candidates": int(load_jsonl(base / "sequences_100.jsonl").shape[0]),
        "final_candidates": int(candidates.shape[0]),
        "correct_at_1": int(bug_success_at_k(validated, "correct", 1).sum()),
        "correct_at_10": int(bug_success_at_k(validated, "correct", 10).sum()),
        "correct_at_100": int(bug_success_at_k(validated, "correct", 100).sum()),
        "plausible_at_1": int(bug_success_at_k(validated, "plausible", 1).sum()),
        "plausible_at_10": int(bug_success_at_k(validated, "plausible", 10).sum()),
        "plausible_at_100": int(bug_success_at_k(validated, "plausible", 100).sum()),
        "correct_bugs": int(bug_success_at_k(validated, "correct", 10**9).sum()),
        "plausible_bugs": int(plausible_bugs.sum()),
        "exact_match_bugs": int(exact_bugs.sum()),
        "truncated_inputs": int(context["was_truncated"].sum()) if context_count else 0,
        "truncate_percent": round(100 * float(context["was_truncated"].sum()) / context_count, 4) if context_count else 0.0,
        "retrieved_contexts": int((context["retrieved_count"] > 0).sum()) if context_count else 0,
        "local_context_at_risk": int(context["local_context_at_risk"].sum()) if context_count else 0,
        "context_records": context_count,
    }

summary = pd.DataFrame([summarize("no_rag"), summarize("fixed_rag")])
for metric in ("correct_at_1", "correct_at_10", "correct_at_100", "correct_bugs", "plausible_bugs", "exact_match_bugs"):
    summary[f"{metric}_rate"] = summary[metric] / summary["bugs"]

def case_status(strategy):
    frame = load_jsonl(output_root / f"outputs-multimend-{strategy}" / "plausible_candidates_100.jsonl")
    frame["plausible"] = frame["plausible"].fillna(False)
    return frame.groupby(["bugid", "hunk"])["plausible"].any()

b0_status = case_status("no_rag")
b1_status = case_status("fixed_rag")
cases = pd.concat([b0_status.rename("b0_plausible"), b1_status.rename("b1_plausible")], axis=1).fillna(False)
cases["classification"] = "unchanged"
cases.loc[~cases.b0_plausible & cases.b1_plausible, "classification"] = "rag_helped"
cases.loc[cases.b0_plausible & ~cases.b1_plausible, "classification"] = "rag_harmed"
cases.reset_index().to_json(output_root / "stage1_rag_case_comparison.jsonl", orient="records", lines=True)
case_counts = cases["classification"].value_counts().to_dict()

summary_path = output_root / "stage1_b0_b1_comparison.json"
summary.to_json(summary_path, orient="records", indent=2)
(output_root / "stage1_rag_case_counts.json").write_text(json.dumps(case_counts, indent=2), encoding="utf-8")
display(summary)
print("RAG case counts:", case_counts)
print("Saved:", summary_path)


,strategy,bugs,hunks,raw_candidates,final_candidates,correct_at_1,correct_at_10,correct_at_100,plausible_at_1,plausible_at_10,...,truncate_percent,retrieved_contexts,local_context_at_risk,context_records,correct_at_1_rate,correct_at_10_rate,correct_at_100_rate,correct_bugs_rate,plausible_bugs_rate,exact_match_bugs_rate
0,no_rag,40,40,20000,9709,5,16,27,7,21,...,0.0,0,0,0,0.125,0.400,0.675,0.675,0.725,0.675
1,fixed_rag,40,40,20000,9667,5,15,27,7,20,...,0.0,0,0,0,0.125,0.375,0.675,0.675,0.725,0.675


RAG case counts: {'unchanged': 40}
Saved: /kaggle/working/MultiMend28/outputs/QuixBugs-Python/stage1_b0_b1_comparison.json


In [142]:
# Cell 9 — Final reproducibility checks and artifact manifest

import hashlib
import json
import subprocess
from datetime import datetime, timezone

def sha256(path):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


run_started_at = datetime.now(timezone.utc).isoformat()
artifact_manifest = {}

SKIPPED_PROGRAMS = {
    "detect_cycle_test",
    "shortest_path_length_test",
}

for strategy in ("no_rag", "fixed_rag"):
    directory = output_root / f"outputs-multimend-{strategy}"

    required = [
        directory / "generated_input.jsonl",
        directory / "sequences_100.jsonl",
        directory / "final_candidates_100.jsonl",
        directory / "plausible_candidates_100.jsonl",
    ]

    missing = [str(path) for path in required if not path.is_file()]
    if missing:
        raise FileNotFoundError(
            f"{strategy} incomplete: {missing}"
        )

    # ---------------------------------------------------------
    # Load raw candidates.
    # load_jsonl() returns a pandas DataFrame.
    # ---------------------------------------------------------
    raw = load_jsonl(
        directory / "sequences_100.jsonl"
    )

    if raw.empty:
        raise RuntimeError(
            f"{strategy}: sequences_100.jsonl is empty"
        )

    # ---------------------------------------------------------
    # Check columns.
    # ---------------------------------------------------------
    required_raw_columns = {"bugid", "hunk"}
    missing_raw_columns = (
        required_raw_columns - set(raw.columns)
    )

    if missing_raw_columns:
        raise RuntimeError(
            f"{strategy}: raw output missing columns: "
            f"{sorted(missing_raw_columns)}"
        )

    # ---------------------------------------------------------
    # Count actual bugs/hunks.
    # Two programs are intentionally skipped.
    # ---------------------------------------------------------
    raw_bugids = set(raw["bugid"].dropna().unique())

    unexpected_skipped = (
        SKIPPED_PROGRAMS & raw_bugids
    )

    if unexpected_skipped:
        raise RuntimeError(
            f"{strategy}: skipped programs unexpectedly appear "
            f"in sequences_100.jsonl: "
            f"{sorted(unexpected_skipped)}"
        )

    hunk_keys = raw[
        ["bugid", "hunk"]
    ].drop_duplicates()

    expected_hunks = len(hunk_keys)
    expected_records = expected_hunks * 5 * 100

    print("=" * 70)
    print(strategy)
    print(f"bugs        : {len(raw_bugids)}")
    print(f"hunks       : {expected_hunks}")
    print(f"raw records : {len(raw)}")
    print(f"expected    : {expected_records}")
    print("=" * 70)

    if len(raw) != expected_records:
        raise RuntimeError(
            f"{strategy}: expected {expected_records} raw records "
            f"for {expected_hunks} hunks, found {len(raw)}"
        )

    # ---------------------------------------------------------
    # Verify validation output.
    # ---------------------------------------------------------
    validated = load_jsonl(
        directory / "plausible_candidates_100.jsonl"
    )

    if validated.empty:
        raise RuntimeError(
            f"{strategy}: plausible_candidates_100.jsonl is empty"
        )

    required_validation_columns = {
        "plausible",
        "correct",
    }

    missing_columns = (
        required_validation_columns
        - set(validated.columns)
    )

    if missing_columns:
        raise RuntimeError(
            f"{strategy}: validation output missing columns: "
            f"{sorted(missing_columns)}"
        )

    artifact_manifest[strategy] = {
        "status": "complete",
        "bugs": len(raw_bugids),
        "hunks": expected_hunks,
        "raw_records": len(raw),
        "validation_records": len(validated),
        "skipped_programs": sorted(SKIPPED_PROGRAMS),
        "files": {
            path.name: sha256(path)
            for path in required
        },
    }


manifest = {
    "stage": 1,
    "dataset": "QuixBugs-Python",
    "strategies": [
        "no_rag",
        "fixed_rag",
    ],
    "seed": 42,
    "num_checkpoints": 5,
    "num_return_sequences": 100,
    "skipped_programs": sorted(SKIPPED_PROGRAMS),
    "rag": {
        "top_k": 5,
        "distance_threshold": 0.5,
    },
    "git_commit": subprocess.check_output(
        ["git", "rev-parse", "HEAD"],
        cwd=REPO,
        text=True,
    ).strip(),
    "run_started_at": run_started_at,
    "run_finished_at": datetime.now(timezone.utc).isoformat(),
    "artifacts": artifact_manifest,
    "comparison_file": str(summary_path),
}


(output_root / "stage1_manifest.json").write_text(
    json.dumps(manifest, indent=2),
    encoding="utf-8",
)

print(json.dumps(manifest, indent=2))
print("STAGE 1 — B0/B1 COMPLETE")


no_rag
bugs        : 40
hunks       : 40
raw records : 20000
expected    : 20000
fixed_rag
bugs        : 40
hunks       : 40
raw records : 20000
expected    : 20000
{
  "stage": 1,
  "dataset": "QuixBugs-Python",
  "strategies": [
    "no_rag",
    "fixed_rag"
  ],
  "seed": 42,
  "num_checkpoints": 5,
  "num_return_sequences": 100,
  "skipped_programs": [
    "detect_cycle_test",
    "shortest_path_length_test"
  ],
  "rag": {
    "top_k": 5,
    "distance_threshold": 0.5
  },
  "git_commit": "c3162d455aa2f7d9e3d02777d461d04d3011d4ca",
  "run_started_at": "2026-10-06T14:14:05.316087+00:00",
  "run_finished_at": "2026-10-06T14:14:05.738729+00:00",
  "artifacts": {
    "no_rag": {
      "status": "complete",
      "bugs": 40,
      "hunks": 40,
      "raw_records": 20000,
      "validation_records": 9709,
      "skipped_programs": [
        "detect_cycle_test",
        "shortest_path_length_test"
      ],
      "files": {
        "generated_input.jsonl": "e37220b998f962f62a9cf9fb6e12eaa

Giai đoạn 0: 

In [ ]:
# Cell 1 — Kaggle environment
import os, sys, subprocess, textwrap, json, pathlib, shutil, time
from pathlib import Path

print("Python:", sys.version)
print("CUDA_VISIBLE_DEVICES:", os.environ.get("CUDA_VISIBLE_DEVICES"))
try:
    import torch
    print("Torch:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
except Exception as e:
    print("Torch import error:", repr(e))


In [ ]:
# Cell 2 — Clone the MultiMend28 fork.
# We intentionally do not recurse through all benchmark submodules yet.

REPO_URL = "https://github.com/ThaoTo2806/MultiMend28.git"
REPO = Path("/kaggle/working/MultiMend28")

if REPO.exists():
    print("Repo already exists:", REPO)
else:
    subprocess.run([
        "git", "clone", "--depth", "1",
        REPO_URL,
        str(REPO)
    ], check=True)

os.chdir(REPO)
print("HEAD:", subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip())
print("Repo:", REPO)
print("Remote:", subprocess.check_output(["git", "remote", "get-url", "origin"], text=True).strip())
print("Top-level:", [p.name for p in REPO.iterdir()])


In [ ]:
# Cell 3 — Inspect the official repo BEFORE running anything
from pathlib import Path

for p in sorted((REPO / "src").rglob("*.py")):
    print(p.relative_to(REPO))

print("\nrequirements.txt:")
print((REPO / "requirements.txt").read_text())


In [ ]:
# Cell 4 — Install official MultiMend dependencies.
# Do NOT upgrade unrelated Kaggle packages.

import subprocess
import sys

result = subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "--no-cache-dir",
        "-r",
        "requirements.txt",
        "more-itertools",
        "pygments",
    ],
    text=True,
)

if result.returncode != 0:
    raise RuntimeError("MultiMend dependency installation failed.")

print("MultiMend dependency installation complete.")

In [ ]:
# Cell 4b — Force the versions required by MultiMend
import subprocess
import sys

packages = [
    "numpy==1.26.4",
    "pandas==2.2.3",
    "torch==2.4.0",
]

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "--no-cache-dir",
        "--force-reinstall",
        *packages,
    ],
    check=True,
)

print("Pinned NumPy/Pandas/Torch versions installed.")

In [ ]:
# Cell 4c — Match torchvision with MultiMend's torch 2.4.0

import subprocess
import sys

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "--no-cache-dir",
        "--force-reinstall",
        "torchvision==0.19.0",
    ],
    check=True,
)

print("torchvision 0.19.0 installed.")

In [ ]:
# Fix NumPy for ChromaDB 0.5.5 / MultiMend
import subprocess
import sys

subprocess.run(
    [
        sys.executable,
        "-m", "pip", "install",
        "-q",
        "--no-cache-dir",
        "--force-reinstall",
        "numpy==1.26.4",
    ],
    check=True,
)

print("NumPy 1.26.4 installed.")


In [ ]:
import numpy as np

print("NumPy:", np.__version__)


In [3]:
import torch
import transformers
import datasets
import chromadb
import sentence_transformers
import tree_sitter
import unidiff
import more_itertools
import pygments

print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("datasets:", datasets.__version__)
print("chromadb:", chromadb.__version__)
print("sentence-transformers:", sentence_transformers.__version__)
print("tree-sitter:", tree_sitter.__version__ if hasattr(tree_sitter, "__version__") else "unknown")
print("CUDA:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


torch: 2.4.0+cu121
transformers: 4.44.2
datasets: 2.21.0
chromadb: 0.5.5
sentence-transformers: 3.0.1
tree-sitter: unknown
CUDA: True
GPU: Tesla T4


In [4]:
# Cell 5 — Verify versions and direct imports
import torch, transformers, datasets, chromadb, sentence_transformers
import tree_sitter, unidiff, more_itertools, pygments

print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("datasets:", datasets.__version__)
print("chromadb:", chromadb.__version__)
print("sentence-transformers:", sentence_transformers.__version__)
print("tree-sitter:", tree_sitter.__version__ if hasattr(tree_sitter, "__version__") else "unknown")
print("CUDA:", torch.cuda.is_available())


torch: 2.4.0+cu121
transformers: 4.44.2
datasets: 2.21.0
chromadb: 0.5.5
sentence-transformers: 3.0.1
tree-sitter: unknown
CUDA: True


In [ ]:
from pathlib import Path

REPO = Path("/kaggle/working/MultiMend28")

print("Repo exists:", REPO.exists())
print("Repo:", REPO)

if REPO.exists():
    print("Contents:")
    for p in list(REPO.iterdir())[:20]:
        print(" -", p.name)


In [ ]:
# Cell 6 — Download QuixBugs

from pathlib import Path
import subprocess
import shutil

REPO = Path("/kaggle/working/MultiMend28")
QB = REPO / "benchmarks" / "QuixBugs"

if not REPO.exists():
    raise FileNotFoundError(
        f"MultiMend repo not found at {REPO}. "
        "Please run the clone cell first."
    )

if not (QB / "python_programs").exists():
    if QB.exists():
        shutil.rmtree(QB)

    subprocess.run([
        "git", "clone", "--depth", "1",
        "https://github.com/jkoppel/QuixBugs.git",
        str(QB)
    ], check=True)

print("QuixBugs files:")
for d in ["python_programs", "correct_python_programs", "python_testcases"]:
    p = QB / d
    print(
        d,
        "=>", p.exists(),
        "count=", len(list(p.glob("*.py"))) if p.exists() else 0
    )


In [ ]:
# ============================================================
# CELL 38 — PREPARE CHECKPOINTS FOR INFERENCE
# ============================================================
# Mục đích:
# - Chuẩn bị 5 checkpoint cuối để inference.
# - Copy checkpoint từ Kaggle Dataset vào thư mục project.
# - Đọc trainer_state.json để xác nhận global_step THỰC TẾ.
# - Không suy luận checkpoint ensemble chỉ từ tên thư mục.
#
# Các checkpoint được sử dụng:
#   348600
#   406700
#   464800
#   522900
#   581008
# ============================================================


import json
import shutil
from pathlib import Path


# ------------------------------------------------------------
# 1. Xác định thư mục project
# ------------------------------------------------------------

REPO = Path(
    "/kaggle/working/MultiMend28"
)


# ------------------------------------------------------------
# 2. Thư mục chứa các checkpoint dùng cho inference
# ------------------------------------------------------------

MODEL_DIR = (
    REPO
    / "models"
    / "multimend-codet5-small"
)

# Tạo thư mục nếu chưa tồn tại
MODEL_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ------------------------------------------------------------
# 3. Khai báo đường dẫn checkpoint nguồn
#
# Key:
#   global step dự kiến
#
# Value:
#   đường dẫn checkpoint trên Kaggle Dataset
# ------------------------------------------------------------

checkpoint_paths = {

    348600:
        "/kaggle/input/datasets/tramtvt/"
        "checkpoint348600/checkpoint-348600",

    406700:
        "/kaggle/input/datasets/tramtvt/"
        "checkpoint406700/checkpoint-406700",

    464800:
        "/kaggle/input/datasets/nguynhongbchtrm/"
        "checkpoint464800/checkpoint-464800",

    522900:
        "/kaggle/input/datasets/nguynhongbchtrm/"
        "checkpoint522900/checkpoint-522900",

    581008:
        "/kaggle/input/datasets/nguynhongbchtrm/"
        "checkpoint581008/checkpoint-581008",
}


# ------------------------------------------------------------
# 4. Copy từng checkpoint
# ------------------------------------------------------------

for step, source_text in checkpoint_paths.items():

    # Đường dẫn checkpoint nguồn
    source = Path(source_text)

    # Đường dẫn checkpoint đích
    target = (
        MODEL_DIR
        / f"checkpoint-{step}"
    )


    # --------------------------------------------------------
    # Kiểm tra checkpoint nguồn có tồn tại
    # --------------------------------------------------------

    if not source.is_dir():

        raise FileNotFoundError(
            f"Không tìm thấy checkpoint: {source}"
        )


    # --------------------------------------------------------
    # Nếu checkpoint đích đã tồn tại:
    # xóa để copy lại từ nguồn cho sạch.
    # --------------------------------------------------------

    if target.exists():

        shutil.rmtree(target)


    # --------------------------------------------------------
    # Copy toàn bộ checkpoint
    # --------------------------------------------------------

    shutil.copytree(
        source,
        target
    )


    # --------------------------------------------------------
    # 5. Đọc trainer_state.json
    #
    # Đây là bước quan trọng:
    # không chỉ tin vào tên checkpoint,
    # mà xác nhận global_step thực tế.
    # --------------------------------------------------------

    state_file = (
        target
        / "trainer_state.json"
    )


    with state_file.open(
        encoding="utf-8"
    ) as file:

        state = json.load(file)


    # --------------------------------------------------------
    # 6. In global_step thực tế
    # --------------------------------------------------------

    print(
        target.name,
        "global_step =",
        state.get("global_step")
    )


# ------------------------------------------------------------
# 7. Hoàn thành bước chuẩn bị checkpoint
# ------------------------------------------------------------

print(
    "Đã chuẩn bị checkpoint inference."
)


In [ ]:
# ============================================================
# CELL 39 — CHECK 5 CHECKPOINTS + MODEL CONFIG
# ============================================================
# Mục đích:
# - Xác nhận 5 checkpoint đã được copy đúng.
# - Đọc config.json của từng checkpoint.
# - Kiểm tra các token ID cần thiết cho CodeT5.
# - Đặc biệt kiểm tra decoder_start_token_id vì lỗi
#   trước đó xảy ra tại model.generate().
# ============================================================

import json
from pathlib import Path

MODEL_DIR = Path(
    "/kaggle/working/MultiMend28/models/multimend-codet5-small"
)

for ckpt in sorted(MODEL_DIR.glob("checkpoint-*")):

    print(f"\n{'=' * 60}")
    print(f"CHECKPOINT: {ckpt.name}")
    print(f"{'=' * 60}")

    config_file = ckpt / "config.json"

    if not config_file.exists():
        print("❌ Missing config.json")
        continue

    with config_file.open(encoding="utf-8") as file:
        config = json.load(file)

    keys = [
        "model_type",
        "architectures",
        "decoder_start_token_id",
        "bos_token_id",
        "eos_token_id",
        "pad_token_id",
        "vocab_size",
    ]

    for key in keys:
        print(f"{key}: {config.get(key)}")


In [ ]:
# ============================================================
# CELL 40 — CHECK TOKENIZER + MODEL
# ============================================================
# Mục đích:
# - Load checkpoint-348600.
# - Kiểm tra tokenizer.
# - Kiểm tra generation config của model.
# - Xác nhận decoder_start_token_id = 0 thực sự được
#   load vào model.
#
# Lưu ý:
# - Không sửa checkpoint.
# - Không chạy inference dataset.
# - Chỉ kiểm tra model trước khi smoke test.
# ============================================================

from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
)

# ------------------------------------------------------------
# 1. Chọn checkpoint dùng để smoke test đầu tiên
# ------------------------------------------------------------

CKPT = (
    MODEL_DIR
    / "checkpoint-348600"
)

print("Loading checkpoint:")
print(CKPT)


# ------------------------------------------------------------
# 2. Load tokenizer
# ------------------------------------------------------------

tokenizer = AutoTokenizer.from_pretrained(
    str(CKPT)
)


# ------------------------------------------------------------
# 3. Load T5 model
# ------------------------------------------------------------

model = AutoModelForSeq2SeqLM.from_pretrained(
    str(CKPT)
)


# ------------------------------------------------------------
# 4. Kiểm tra tokenizer
# ------------------------------------------------------------

print("\n========== TOKENIZER ==========")

print(
    "pad_token    =",
    tokenizer.pad_token
)

print(
    "pad_token_id =",
    tokenizer.pad_token_id
)

print(
    "bos_token    =",
    tokenizer.bos_token
)

print(
    "bos_token_id =",
    tokenizer.bos_token_id
)

print(
    "eos_token    =",
    tokenizer.eos_token
)

print(
    "eos_token_id =",
    tokenizer.eos_token_id
)


# ------------------------------------------------------------
# 5. Kiểm tra model.config
# ------------------------------------------------------------

print("\n========== MODEL CONFIG ==========")

print(
    "model_type =",
    model.config.model_type
)

print(
    "architectures =",
    model.config.architectures
)

print(
    "decoder_start_token_id =",
    model.config.decoder_start_token_id
)

print(
    "bos_token_id =",
    model.config.bos_token_id
)

print(
    "eos_token_id =",
    model.config.eos_token_id
)

print(
    "pad_token_id =",
    model.config.pad_token_id
)


# ------------------------------------------------------------
# 6. Kiểm tra generation_config
#
# Một số phiên bản Transformers lấy các giá trị generation
# từ model.generation_config thay vì trực tiếp từ model.config.
# Đây có thể là nguyên nhân gây lỗi trước đó.
# ------------------------------------------------------------

print("\n========== GENERATION CONFIG ==========")

print(
    "decoder_start_token_id =",
    model.generation_config.decoder_start_token_id
)

print(
    "bos_token_id =",
    model.generation_config.bos_token_id
)

print(
    "eos_token_id =",
    model.generation_config.eos_token_id
)

print(
    "pad_token_id =",
    model.generation_config.pad_token_id
)


In [ ]:
# ============================================================
# CELL 41 — FIX / SYNC GENERATION CONFIG
# ============================================================
# Mục đích:
# - Không thay đổi file checkpoint trên disk.
# - Đồng bộ các token ID từ model.config sang
#   model.generation_config.
# - Khắc phục lỗi:
#   "decoder_start_token_id or bos_token_id has to be defined"
#
# Nguyên nhân đã xác định:
#   model.config.decoder_start_token_id = 0
#   model.generation_config.decoder_start_token_id = None
#
# Vì transformers.generate() sử dụng generation_config,
# cần đồng bộ hai config trước khi generate().
# ============================================================

# ------------------------------------------------------------
# Đồng bộ decoder_start_token_id
# ------------------------------------------------------------

model.generation_config.decoder_start_token_id = (
    model.config.decoder_start_token_id
)


# ------------------------------------------------------------
# Đồng bộ các token ID còn lại
# ------------------------------------------------------------

model.generation_config.bos_token_id = (
    model.config.bos_token_id
)

model.generation_config.eos_token_id = (
    model.config.eos_token_id
)

model.generation_config.pad_token_id = (
    model.config.pad_token_id
)


# ------------------------------------------------------------
# In lại để xác nhận
# ------------------------------------------------------------

print("========== GENERATION CONFIG AFTER SYNC ==========")

print(
    "decoder_start_token_id =",
    model.generation_config.decoder_start_token_id
)

print(
    "bos_token_id =",
    model.generation_config.bos_token_id
)

print(
    "eos_token_id =",
    model.generation_config.eos_token_id
)

print(
    "pad_token_id =",
    model.generation_config.pad_token_id
)


In [ ]:
# ============================================================
# CELL 42 — SMOKE TEST MODEL.GENERATE()
# ============================================================
# Mục đích:
# - Kiểm tra trực tiếp model.generate().
# - Chưa chạy QuixBugs.
# - Chưa chạy cả 5 checkpoint.
# - Nếu cell này PASS thì nguyên nhân lỗi generation đã được
#   xử lý.
# ============================================================

import torch

# ------------------------------------------------------------
# Chọn GPU nếu Kaggle có CUDA, ngược lại dùng CPU
# ------------------------------------------------------------

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("Device:", device)


# ------------------------------------------------------------
# Đưa model sang device và chuyển sang evaluation mode
# ------------------------------------------------------------

model = model.to(device)
model.eval()


# ------------------------------------------------------------
# Input đơn giản để kiểm tra CodeT5
# ------------------------------------------------------------

test_input = (
    "fix bug: def add(a, b): return a - b"
)

print("\nInput:")
print(test_input)


# ------------------------------------------------------------
# Tokenize input
# ------------------------------------------------------------

inputs = tokenizer(
    test_input,
    return_tensors="pt",
    truncation=True,
    max_length=512,
)


# ------------------------------------------------------------
# Đưa tensors lên cùng device với model
# ------------------------------------------------------------

inputs = {
    key: value.to(device)
    for key, value in inputs.items()
}


# ------------------------------------------------------------
# Generate output
# ------------------------------------------------------------

with torch.no_grad():

    outputs = model.generate(
        **inputs,

        # GenerationConfig đã được đồng bộ ở Cell 41
        max_length=128,

        # Baseline deterministic
        do_sample=False,
    )


# ------------------------------------------------------------
# Decode prediction
# ------------------------------------------------------------

generated_text = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True,
)


# ------------------------------------------------------------
# In kết quả
# ------------------------------------------------------------

print("\nGenerated output:")
print(generated_text)

print("\n✅ model.generate() smoke test PASSED")


In [ ]:
# ============================================================
# CELL 43 — INSPECT QUIXBUGS INPUT FORMAT
# ============================================================
# Mục đích:
# - Không tự đoán format input của model.
# - Kiểm tra chính xác dữ liệu QuixBugs đang được pipeline
#   sử dụng.
# - Xác định input column và cách preprocessing.
#
# QUAN TRỌNG:
# - Chưa chạy model.
# - Chỉ đọc dữ liệu và in một vài sample.
# ============================================================

from pathlib import Path
import json

REPO = Path(
    "/kaggle/working/MultiMend28"
)

print("Repository:", REPO)

# ------------------------------------------------------------
# Tìm các file dữ liệu có khả năng là QuixBugs
# ------------------------------------------------------------

candidates = []

for pattern in [
    "*.json",
    "*.jsonl",
    "*.arrow",
    "*.parquet",
    "*.csv",
]:

    for path in REPO.rglob(pattern):

        name = path.name.lower()

        if (
            "quix" in name
            or "test" in name
            or "bug" in name
        ):
            candidates.append(path)

print("\nPossible dataset files:")

for path in candidates[:50]:
    print(path)


In [ ]:
# ============================================================
# CELL 44 — INSPECT INFERENCE SCRIPT
# ============================================================
# Mục đích:
# - Đọc generate_primary_candidates.py.
# - Tìm:
#     + cách load dataset
#     + preprocessing/tokenization
#     + input column
#     + prompt/template
#     + model.generate()
# - Không sửa file.
# ============================================================

from pathlib import Path

script_path = (
    Path("/kaggle/working/MultiMend28")
    / "src"
    / "generate_primary_candidates.py"
)

print("Script:")
print(script_path)

if not script_path.exists():
    raise FileNotFoundError(script_path)

text = script_path.read_text(
    encoding="utf-8"
)

# ------------------------------------------------------------
# In toàn bộ script để kiểm tra pipeline.
# Nếu script dài, có thể xem phần liên quan generate_candidates
# trước.
# ------------------------------------------------------------

print(text)


In [ ]:
# ============================================================
# CELL 45 — INSPECT INFERENCE / GENERATION PIPELINE
# ============================================================
# Mục đích:
# - Kiểm tra chính xác cách project:
#     + chọn dataset QuixBugs-Python
#     + tạo input
#     + load checkpoint
#     + gọi model.generate()
#     + ghi output
#
# - Không chạy inference.
# - Không sửa source code.
#
# Đây là bước cần thiết trước khi sửa generate_primary_candidates.py
# vì ta phải giữ nguyên preprocessing của project.
# ============================================================

from pathlib import Path

script_path = (
    Path("/kaggle/working/MultiMend28")
    / "src"
    / "generate_primary_candidates.py"
)

lines = script_path.read_text(
    encoding="utf-8"
).splitlines()

# ------------------------------------------------------------
# In các dòng từ phần tokenize/generation trở đi.
# ------------------------------------------------------------

for line_number, line in enumerate(
    lines[80:260],
    start=81
):
    print(
        f"{line_number:4d}: {line}"
    )


In [ ]:
# ============================================================
# CELL 46 — PATCH PRIMARY INFERENCE SCRIPT
# ============================================================
# Mục đích:
# 1. Backup generate_primary_candidates.py trước khi sửa.
#
# 2. Sửa lỗi GenerationConfig:
#       model.config.decoder_start_token_id = 0
#       nhưng
#       model.generation_config.decoder_start_token_id = None
#
# 3. Đảm bảo 5 checkpoint cuối được chọn dựa trên
#    trainer_state.json -> global_step,
#    KHÔNG chỉ dựa vào tên thư mục.
#
# 4. Không thay đổi checkpoint trên disk.
# ============================================================

from pathlib import Path
import shutil

REPO = Path(
    "/kaggle/working/MultiMend28"
)

SCRIPT = (
    REPO
    / "src"
    / "generate_primary_candidates.py"
)

BACKUP = (
    REPO
    / "src"
    / "generate_primary_candidates.py.bak"
)


# ------------------------------------------------------------
# 1. Kiểm tra source script
# ------------------------------------------------------------

if not SCRIPT.exists():
    raise FileNotFoundError(
        f"Không tìm thấy: {SCRIPT}"
    )


# ------------------------------------------------------------
# 2. Backup script gốc
# ------------------------------------------------------------

if not BACKUP.exists():
    shutil.copy2(
        SCRIPT,
        BACKUP
    )

    print(
        "Đã backup script gốc:"
    )
    print(BACKUP)

else:
    print(
        "Backup đã tồn tại:"
    )
    print(BACKUP)


# ------------------------------------------------------------
# 3. Đọc source code
# ------------------------------------------------------------

text = SCRIPT.read_text(
    encoding="utf-8"
)


# ============================================================
# PATCH A — chọn checkpoint theo trainer_state.json
# ============================================================

old_get_checkpoints = '''def get_checkpoints(checkpoints_dir: Path) -> list[tuple[str, Path]]:
    """Get the list of checkpoints from the checkpoints directory"""

    checkpoints: dict[str, Path] = {}
    for d in checkpoints_dir.iterdir():
        name_parts = d.name.split("-")
        if (
            len(name_parts) == 2
            and name_parts[0] == "checkpoint"
            and name_parts[1].isdigit()
        ):
            checkpoints[d.name] = d

    sorted_checkpoints = sorted(
        checkpoints.items(), key=lambda x: int(x[0].split("-")[1])
    )

    return sorted_checkpoints
'''

new_get_checkpoints = '''def get_checkpoints(checkpoints_dir: Path) -> list[tuple[str, Path]]:
    """Get checkpoints sorted by trainer_state.json global_step."""

    checkpoints = []

    for d in checkpoints_dir.iterdir():
        name_parts = d.name.split("-")

        if not (
            len(name_parts) == 2
            and name_parts[0] == "checkpoint"
            and name_parts[1].isdigit()
        ):
            continue

        state_file = d / "trainer_state.json"

        if not state_file.exists():
            print(f"WARNING: missing trainer_state.json: {d}")
            continue

        with state_file.open(encoding="utf-8") as file:
            state = json.load(file)

        global_step = state.get("global_step")

        if global_step is None:
            print(f"WARNING: missing global_step: {d}")
            continue

        checkpoints.append(
            (d.name, d, int(global_step))
        )

    checkpoints.sort(key=lambda x: x[2])

    return [
        (name, path)
        for name, path, _ in checkpoints
    ]
'''

if old_get_checkpoints not in text:
    raise RuntimeError(
        "Không tìm thấy hàm get_checkpoints() "
        "theo đúng nội dung dự kiến."
    )

text = text.replace(
    old_get_checkpoints,
    new_get_checkpoints
)


# ============================================================
# PATCH B — sync GenerationConfig sau khi load model
# ============================================================

old_model_load = '''    model = AutoModelForSeq2SeqLM.from_pretrained(checkpoint).to(device)
    results = tokenized_test_dataset.map(
'''

new_model_load = '''    model = AutoModelForSeq2SeqLM.from_pretrained(checkpoint).to(device)

    # --------------------------------------------------------
    # Sync GenerationConfig với model.config.
    #
    # Một số checkpoint có decoder_start_token_id trong
    # model.config nhưng generation_config lại để None.
    # transformers.generate() sử dụng generation_config,
    # nên cần đồng bộ trước khi generate().
    # --------------------------------------------------------

    model.generation_config.decoder_start_token_id = (
        model.config.decoder_start_token_id
    )

    model.generation_config.bos_token_id = (
        model.config.bos_token_id
    )

    model.generation_config.eos_token_id = (
        model.config.eos_token_id
    )

    model.generation_config.pad_token_id = (
        model.config.pad_token_id
    )

    if model.generation_config.decoder_start_token_id is None:
        raise ValueError(
            f"Checkpoint {ch_name} has no "
            "decoder_start_token_id."
        )

    print(
        "decoder_start_token_id =",
        model.generation_config.decoder_start_token_id
    )

    results = tokenized_test_dataset.map(
'''

if old_model_load not in text:
    raise RuntimeError(
        "Không tìm thấy đoạn load model "
        "theo đúng nội dung dự kiến."
    )

text = text.replace(
    old_model_load,
    new_model_load
)


# ------------------------------------------------------------
# 4. Ghi source code đã sửa
# ------------------------------------------------------------

SCRIPT.write_text(
    text,
    encoding="utf-8"
)

print("\nĐã patch:")
print(SCRIPT)


In [ ]:
# ============================================================
# CELL 47 — VERIFY INFERENCE SCRIPT PATCH
# ============================================================
# Mục đích:
# - Xác nhận get_checkpoints() đã đọc trainer_state.json.
# - Xác nhận GenerationConfig được sync.
# - Không chạy model inference.
# ============================================================

from pathlib import Path

SCRIPT = Path(
    "/kaggle/working/MultiMend28/src/"
    "generate_primary_candidates.py"
)

text = SCRIPT.read_text(
    encoding="utf-8"
)


# ------------------------------------------------------------
# Kiểm tra PATCH A
# ------------------------------------------------------------

assert (
    'state_file = d / "trainer_state.json"'
    in text
)

assert (
    'global_step = state.get("global_step")'
    in text
)


# ------------------------------------------------------------
# Kiểm tra PATCH B
# ------------------------------------------------------------

assert (
    "model.generation_config.decoder_start_token_id"
    in text
)

assert (
    "model.config.decoder_start_token_id"
    in text
)


print(
    "✅ Checkpoint selection patch: OK"
)

print(
    "✅ GenerationConfig patch: OK"
)

print(
    "✅ Source verification PASSED"
)


In [ ]:
# ============================================================
# CELL 48 — QUIXBUGS-PYTHON SMOKE TEST
# ============================================================
# Mục đích:
# - Chạy pipeline inference thật của project.
# - Dataset: QuixBugs-Python.
# - Seed trong script: 42.
# - Sử dụng 5 checkpoint cuối theo trainer_state.json.
# - Giữ nguyên beam_size = 100.
# - Giữ nguyên num_return_sequences = 100.
#
# Lưu ý:
# - Pipeline hiện tại sẽ chạy 5 checkpoint.
# - Với 40 mẫu và 100 candidates/checkpoint,
#   số lượng output sẽ khá lớn.
# ============================================================

import subprocess
from pathlib import Path

REPO = Path(
    "/kaggle/working/MultiMend28"
)

command = [
    "/usr/bin/python3",
    "-m",
    "src.generate_primary_candidates",
]

print(
    "$",
    " ".join(command)
)

subprocess.run(
    command,
    cwd=REPO,
    check=True,
)

print(
    "\n✅ QuixBugs-Python inference completed."
)


In [ ]:
# ============================================================
# CELL 49 — VERIFY QUIXBUGS JSONL OUTPUT
# ============================================================
# Mục đích:
# - Tìm file JSONL được sinh bởi inference.
# - Kiểm tra JSONL có đọc được hay không.
# - Kiểm tra các field chính:
#       checkpoint
#       decoded_sequences
#       sequences_scores
#       bugid
#       hunk
# - Đếm số dòng output.
#
# Với:
#   40 mẫu
#   5 checkpoint
#   100 candidates/checkpoint
#
# kỳ vọng khoảng:
#   40 × 5 = 200 records
#
# Mỗi record chứa 100 decoded_sequences.
# ============================================================

import json
from pathlib import Path

REPO = Path(
    "/kaggle/working/MultiMend28"
)

# ------------------------------------------------------------
# Thư mục output của QuixBugs-Python
# ------------------------------------------------------------

output_root = REPO

jsonl_files = sorted(
    output_root.rglob("sequences_*.jsonl")
)

print("Found JSONL files:")

for path in jsonl_files:
    print(" -", path)


# ------------------------------------------------------------
# Kiểm tra từng JSONL
# ------------------------------------------------------------

for path in jsonl_files:

    print("\n" + "=" * 70)
    print("FILE:", path)
    print("=" * 70)

    valid_lines = 0
    invalid_lines = 0

    checkpoints = set()
    bugids = set()

    first_record = None

    with path.open(
        "r",
        encoding="utf-8"
    ) as file:

        for line_number, line in enumerate(
            file,
            start=1
        ):

            line = line.strip()

            if not line:
                continue

            try:
                record = json.loads(line)

            except json.JSONDecodeError as exc:

                invalid_lines += 1

                print(
                    f"❌ Invalid JSON at line "
                    f"{line_number}: {exc}"
                )

                continue

            valid_lines += 1

            # Lưu record đầu tiên để inspect schema
            if first_record is None:
                first_record = record

            # Thu thập checkpoint
            if "checkpoint" in record:

                checkpoints.update(
                    record["checkpoint"]
                )

            # Thu thập bug IDs
            if "bugid" in record:

                bugids.update(
                    record["bugid"]
                )


    print("\nValid JSON lines  :", valid_lines)
    print("Invalid JSON lines:", invalid_lines)

    print(
        "\nCheckpoints found:"
    )

    for checkpoint in sorted(checkpoints):
        print(" -", checkpoint)

    print(
        "\nNumber of unique bugids:",
        len(bugids)
    )


    # --------------------------------------------------------
    # Inspect record đầu tiên
    # --------------------------------------------------------

    if first_record is not None:

        print(
            "\nFirst record keys:"
        )

        print(
            list(first_record.keys())
        )

        print(
            "\nFirst record:"
        )

        print(
            json.dumps(
                first_record,
                indent=2,
                ensure_ascii=False
            )[:5000]
        )


In [ ]:
# ============================================================
# CELL 50 — VALIDATE BASELINE OUTPUT SCHEMA
# ============================================================
# Mục đích:
# - Kiểm tra chính xác 20,000 candidate.
# - Xác nhận đủ 5 checkpoint.
# - Xác nhận mỗi checkpoint có 40 × 100 candidates.
# - Xác nhận đủ 40 bug/hunk.
# - Kiểm tra JSONL không bị lỗi.
#
# Kỳ vọng:
#
#   40 bugs
#   × 5 checkpoints
#   × 100 candidates
#   = 20,000 JSONL records
#
# Không thực hiện ensemble.
# Không thực hiện patch ranking.
# ============================================================

import json
from collections import Counter
from pathlib import Path


# ------------------------------------------------------------
# 1. Xác định output JSONL
# ------------------------------------------------------------

OUTPUT = Path(
    "/kaggle/working/MultiMend28/"
    "outputs/QuixBugs-Python/"
    "outputs-multimend/"
    "sequences_100.jsonl"
)

if not OUTPUT.exists():
    raise FileNotFoundError(
        f"Không tìm thấy output: {OUTPUT}"
    )


# ------------------------------------------------------------
# 2. Đọc JSONL
# ------------------------------------------------------------

records = []

with OUTPUT.open(
    "r",
    encoding="utf-8"
) as file:

    for line_number, line in enumerate(
        file,
        start=1
    ):

        line = line.strip()

        if not line:
            continue

        record = json.loads(line)

        records.append(record)


# ------------------------------------------------------------
# 3. Tổng số candidate
# ------------------------------------------------------------

print("Total JSONL records:", len(records))


# ------------------------------------------------------------
# 4. Kiểm tra schema
# ------------------------------------------------------------

required_keys = {
    "checkpoint",
    "decoded_sequences",
    "sequences_scores",
    "bugid",
    "hunk",
}

for i, record in enumerate(records):

    missing = (
        required_keys
        - set(record.keys())
    )

    if missing:

        raise ValueError(
            f"Record {i} thiếu field: {missing}"
        )


print(
    "Schema validation: PASS"
)


# ------------------------------------------------------------
# 5. Đếm checkpoint
# ------------------------------------------------------------

checkpoint_counts = Counter(
    record["checkpoint"]
    for record in records
)

print("\nCheckpoint counts:")

for checkpoint, count in sorted(
    checkpoint_counts.items()
):

    print(
        f"  {checkpoint}: {count}"
    )


# ------------------------------------------------------------
# 6. Đếm bug
# ------------------------------------------------------------

bug_counts = Counter(
    record["bugid"]
    for record in records
)

print(
    "\nUnique bugids:",
    len(bug_counts)
)

print(
    "Bug/hunk combinations:",
    len({
        (
            record["bugid"],
            record["hunk"]
        )
        for record in records
    })
)


# ------------------------------------------------------------
# 7. Kiểm tra 5 checkpoint
# ------------------------------------------------------------

expected_checkpoints = {
    "checkpoint-348600",
    "checkpoint-406700",
    "checkpoint-464800",
    "checkpoint-522900",
    "checkpoint-581008",
}

actual_checkpoints = set(
    checkpoint_counts.keys()
)

assert actual_checkpoints == expected_checkpoints, (
    "Checkpoint không đúng.\n"
    f"Expected: {expected_checkpoints}\n"
    f"Actual:   {actual_checkpoints}"
)


# ------------------------------------------------------------
# 8. Mỗi checkpoint phải có 4,000 candidates
#
# 40 inputs × 100 candidates = 4,000
# ------------------------------------------------------------

for checkpoint in expected_checkpoints:

    assert (
        checkpoint_counts[checkpoint] == 4000
    ), (
        f"{checkpoint} có "
        f"{checkpoint_counts[checkpoint]} records, "
        f"không phải 4000."
    )


# ------------------------------------------------------------
# 9. Tổng số candidate
# ------------------------------------------------------------

assert len(records) == 20_000, (
    f"Expected 20,000 records, "
    f"got {len(records)}"
)


# ------------------------------------------------------------
# 10. Kiểm tra score có tồn tại
# ------------------------------------------------------------

missing_scores = sum(
    record["sequences_scores"] is None
    for record in records
)

print(
    "\nMissing sequence scores:",
    missing_scores
)

assert missing_scores == 0


# ------------------------------------------------------------
# 11. In một vài candidate mẫu
# ------------------------------------------------------------

print("\nSample candidates:")

for record in records[:10]:

    print(
        record["checkpoint"],
        "|",
        record["bugid"],
        "| hunk =",
        record["hunk"],
        "| score =",
        record["sequences_scores"],
        "|",
        repr(record["decoded_sequences"])
    )


# ------------------------------------------------------------
# Kết luận
# ------------------------------------------------------------

print(
    "\n" + "=" * 60
)

print(
    "✅ BASELINE JSONL VALIDATION PASSED"
)

print(
    "20,000 candidates"
)

print(
    "5 checkpoints × 40 inputs × 100 candidates"
)

print(
    "No checkpoint ensemble performed."
)

print(
    "=" * 60
)


In [ ]:
# ============================================================
# CELL 51 — BASELINE MANIFEST / CONFIG HASH
# ============================================================
# Mục đích:
# - Ghi lại đầy đủ cấu hình của baseline inference.
# - Seed cố định = 42.
# - Ghi chính xác 5 checkpoint được sử dụng.
# - Xác nhận global_step từ trainer_state.json.
# - Tạo SHA256 hash cho configuration.
# - Ghi input/output JSONL.
#
# Lưu ý:
# - Không chạy inference.
# - Không ensemble.
# - Không thay đổi checkpoint.
# ============================================================

import hashlib
import json
from pathlib import Path


# ------------------------------------------------------------
# 1. Paths
# ------------------------------------------------------------

REPO = Path(
    "/kaggle/working/MultiMend28"
)

MODEL_DIR = (
    REPO
    / "models"
    / "multimend-codet5-small"
)

OUTPUT_DIR = (
    REPO
    / "outputs"
    / "QuixBugs-Python"
    / "outputs-multimend"
)

INPUT_JSONL = (
    OUTPUT_DIR
    / "generated_input.jsonl"
)

OUTPUT_JSONL = (
    OUTPUT_DIR
    / "sequences_100.jsonl"
)


# ------------------------------------------------------------
# 2. Baseline configuration
# ------------------------------------------------------------

config = {
    "dataset": "QuixBugs-Python",
    "model_name": "multimend-codet5-small",

    "seed": 42,

    "max_input_length": 512,
    "max_target_length": 256,
    "min_target_length": 0,

    "beam_size": 100,
    "num_return_sequences": 100,
    "batch_size": 1,
    "num_checkpoints": 5,

    "device": (
        "cuda"
        if __import__("torch").cuda.is_available()
        else "cpu"
    ),

    "input_jsonl": str(INPUT_JSONL),
    "output_jsonl": str(OUTPUT_JSONL),
}


# ------------------------------------------------------------
# 3. Đọc 5 checkpoint và trainer_state.json
# ------------------------------------------------------------

checkpoint_info = []

for checkpoint_dir in sorted(
    MODEL_DIR.glob("checkpoint-*"),
    key=lambda p: int(p.name.split("-")[1])
):

    state_file = (
        checkpoint_dir
        / "trainer_state.json"
    )

    if not state_file.exists():
        continue

    with state_file.open(
        encoding="utf-8"
    ) as file:

        state = json.load(file)

    global_step = state.get(
        "global_step"
    )

    checkpoint_info.append(
        {
            "name": checkpoint_dir.name,
            "path": str(checkpoint_dir),
            "global_step": global_step,
        }
    )


# ------------------------------------------------------------
# 4. Chọn 5 checkpoint cuối theo global_step
# ------------------------------------------------------------

checkpoint_info.sort(
    key=lambda x: x["global_step"]
)

selected_checkpoints = (
    checkpoint_info[-5:]
)

config["checkpoints"] = [
    {
        "name": item["name"],
        "global_step": item["global_step"],
    }
    for item in selected_checkpoints
]


# ------------------------------------------------------------
# 5. Tạo canonical JSON
#
# sort_keys=True giúp hash ổn định.
# ------------------------------------------------------------

canonical_config = json.dumps(
    config,
    sort_keys=True,
    separators=(",", ":"),
    ensure_ascii=False,
)


# ------------------------------------------------------------
# 6. SHA256 config hash
# ------------------------------------------------------------

config_hash = hashlib.sha256(
    canonical_config.encode("utf-8")
).hexdigest()

config["config_hash"] = (
    f"sha256:{config_hash}"
)


# ------------------------------------------------------------
# 7. Tạo manifest
# ------------------------------------------------------------

manifest = {
    "experiment": "MultiMend28 baseline inference",

    "config": config,

    "selected_checkpoints": selected_checkpoints,

    "input": {
        "path": str(INPUT_JSONL),
        "exists": INPUT_JSONL.exists(),
    },

    "output": {
        "path": str(OUTPUT_JSONL),
        "exists": OUTPUT_JSONL.exists(),
    },
}


# ------------------------------------------------------------
# 8. Lưu manifest
# ------------------------------------------------------------

MANIFEST = (
    OUTPUT_DIR
    / "baseline_manifest.json"
)

with MANIFEST.open(
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        manifest,
        file,
        indent=2,
        ensure_ascii=False,
    )


# ------------------------------------------------------------
# 9. In thông tin quan trọng
# ------------------------------------------------------------

print("=" * 70)
print("BASELINE MANIFEST")
print("=" * 70)

print(
    "\nDataset:",
    config["dataset"]
)

print(
    "Model:",
    config["model_name"]
)

print(
    "Seed:",
    config["seed"]
)

print(
    "Config hash:",
    config["config_hash"]
)

print(
    "\nSelected checkpoints:"
)

for checkpoint in selected_checkpoints:

    print(
        f"  {checkpoint['name']} "
        f"(global_step={checkpoint['global_step']})"
    )

print(
    "\nInput JSONL:",
    INPUT_JSONL
)

print(
    "Output JSONL:",
    OUTPUT_JSONL
)

print(
    "\nManifest:",
    MANIFEST
)

print(
    "\n✅ Baseline manifest created."
)


In [ ]:
# ============================================================
# CELL 52 — INSPECT OFFICIAL QUIXBUGS-PYTHON VALIDATOR
# ============================================================
# Mục đích:
# - Đọc validator chính thức của MultiMend28.
# - Xác định:
#     + input arguments
#     + output arguments
#     + cách apply candidate patch
#     + cách chạy test
#     + cách ghi kết quả
#
# - Không chạy evaluation ở cell này.
# ============================================================

from pathlib import Path

REPO = Path(
    "/kaggle/working/MultiMend28"
)

VALIDATOR = (
    REPO
    / "src"
    / "validators"
    / "validate_quixbugs_python.py"
)

if not VALIDATOR.exists():
    raise FileNotFoundError(
        f"Không tìm thấy validator: {VALIDATOR}"
    )

print("=" * 70)
print("OFFICIAL QUIXBUGS-PYTHON VALIDATOR")
print("=" * 70)

print(
    VALIDATOR.read_text(
        encoding="utf-8"
    )
)


In [ ]:
# ============================================================
# CELL 53 — FIND FINAL CANDIDATE PIPELINE
# ============================================================
# Mục đích:
# - Tìm script tạo final_candidates_100.jsonl.
# - Xác định bước ranking/filter candidate của MultiMend28.
# - Không chạy evaluation ở cell này.
#
# Luồng chính thức dự kiến:
#
#   sequences_100.jsonl
#          ↓
#   candidate selection / ranking
#          ↓
#   final_candidates_100.jsonl
#          ↓
#   validate_quixbugs_python.py
#          ↓
#   plausible_candidates_100.jsonl
# ============================================================

from pathlib import Path

REPO = Path(
    "/kaggle/working/MultiMend28"
)

OUTPUT_DIR = (
    REPO
    / "outputs"
    / "QuixBugs-Python"
    / "outputs-multimend"
)

print("=" * 70)
print("EXISTING CANDIDATE OUTPUTS")
print("=" * 70)

for path in sorted(OUTPUT_DIR.glob("*.jsonl")):
    print(path)

print("\n" + "=" * 70)
print("SCRIPTS MENTIONING final_candidates")
print("=" * 70)

for path in REPO.rglob("*.py"):

    try:
        text = path.read_text(
            encoding="utf-8"
        )
    except Exception:
        continue

    if "final_candidates" in text:
        print(path)


In [ ]:
# ============================================================
# CELL 54 — INSPECT CHECKPOINT COMBINATION SCRIPT
# ============================================================
# Mục đích:
# - Đọc script chính thức combine kết quả từ nhiều checkpoint.
# - Xác định chính xác:
#     + cách gộp 5 checkpoint
#     + cách ranking candidate
#     + cách loại duplicate
#     + cách tạo final_candidates_100.jsonl
#
# Lưu ý:
# - Chưa chạy script.
# - Chưa validation patch.
# - Chưa thay đổi sequences_100.jsonl.
# ============================================================

from pathlib import Path

REPO = Path(
    "/kaggle/working/MultiMend28"
)

SCRIPT = (
    REPO
    / "src"
    / "combine_checkpoints_results.py"
)

if not SCRIPT.exists():
    raise FileNotFoundError(
        f"Không tìm thấy: {SCRIPT}"
    )

print("=" * 70)
print("COMBINE CHECKPOINTS SCRIPT")
print("=" * 70)

print(
    SCRIPT.read_text(
        encoding="utf-8"
    )
)


In [ ]:
# ============================================================
# CELL 55 — COMBINE CHECKPOINT RESULTS
# ============================================================
# Mục đích:
# - Dùng pipeline chính thức của MultiMend28.
# - Gộp candidate từ 5 checkpoint đã inference.
# - Normalize và deduplicate candidate patches.
# - Tạo exact_match flag.
#
# Input:
#   sequences_100.jsonl
#
# Output:
#   final_candidates_100.jsonl
#
# Quan trọng:
# - Không train lại model.
# - Không chạy model.generate().
# - Không ensemble xác suất/weights giữa checkpoint.
# - Không chọn checkpoint dựa trên tên thư mục.
# - 5 checkpoint đã được xác nhận bằng trainer_state.json.
# ============================================================

import subprocess
import sys
from pathlib import Path


REPO = Path(
    "/kaggle/working/MultiMend28"
)

INPUT = (
    REPO
    / "outputs"
    / "QuixBugs-Python"
    / "outputs-multimend"
    / "sequences_100.jsonl"
)

OUTPUT = (
    REPO
    / "outputs"
    / "QuixBugs-Python"
    / "outputs-multimend"
    / "final_candidates_100.jsonl"
)


# ------------------------------------------------------------
# Kiểm tra input
# ------------------------------------------------------------

if not INPUT.exists():
    raise FileNotFoundError(
        f"Không tìm thấy input: {INPUT}"
    )


# ------------------------------------------------------------
# Chạy script chính thức
# ------------------------------------------------------------

print("=" * 70)
print("Running official checkpoint combination")
print("=" * 70)

command = [
    sys.executable,
    "-m",
    "src.combine_checkpoints_results",
]

print(
    "$",
    " ".join(command)
)

subprocess.run(
    command,
    cwd=REPO,
    check=True,
)


# ------------------------------------------------------------
# Kiểm tra output
# ------------------------------------------------------------

if not OUTPUT.exists():
    raise FileNotFoundError(
        f"Không tạo được output: {OUTPUT}"
    )


print("\n" + "=" * 70)
print("COMBINATION COMPLETED")
print("=" * 70)

print(
    "Input :",
    INPUT
)

print(
    "Output:",
    OUTPUT
)

print(
    "\n✅ final_candidates_100.jsonl created."
)


In [ ]:
# ============================================================
# CELL 56 — VALIDATE FINAL CANDIDATES
# ============================================================
# Mục đích:
# - Kiểm tra final_candidates_100.jsonl sau khi combine.
# - Xác nhận JSONL hợp lệ.
# - Kiểm tra đủ 9,667 candidate records.
# - Kiểm tra schema.
# - Kiểm tra candidate thực tế có nội dung.
# - Hiển thị candidate samples.
# - Kiểm tra đủ 40 bug/hunk.
# - Kiểm tra phân bố candidate theo checkpoint.
# - Thống kê exact_match và empty patches.
#
# PHẠM VI STAGE 0:
# - Artifact validation.
# - Candidate validation.
# - KHÔNG chạy pytest.
# - KHÔNG chạy official repair validator.
# - KHÔNG đánh giá plausible/parsable.
# - KHÔNG tính repair rate.
# - KHÔNG thực hiện checkpoint ensemble.
# ============================================================

import json
from collections import Counter
from pathlib import Path


# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

OUTPUT = Path(
    "/kaggle/working/MultiMend28/"
    "outputs/QuixBugs-Python/"
    "outputs-multimend/"
    "final_candidates_100.jsonl"
)

EXPECTED_RECORDS = 9667
EXPECTED_RAW_CANDIDATES = 20000
EXPECTED_BUGIDS = 40
EXPECTED_BUG_HUNKS = 40
EXPECTED_SEED = 42

EXPECTED_CONFIG_HASH = (
    "sha256:"
    "c0c516a4f019eabf3b648e39a901e7b81571636ba97dbc31ce8d64063962eaaa"
)

EXPECTED_CHECKPOINTS = {
    "checkpoint-348600",
    "checkpoint-406700",
    "checkpoint-464800",
    "checkpoint-522900",
    "checkpoint-581008",
    "manual",
}


# ------------------------------------------------------------
# Header
# ------------------------------------------------------------

print("=" * 70)
print("STAGE 0 — FINAL CANDIDATES VALIDATION")
print("=" * 70)


# ------------------------------------------------------------
# Check file
# ------------------------------------------------------------

assert OUTPUT.exists(), (
    f"Không tìm thấy file: {OUTPUT}"
)

print("\nInput:")
print(OUTPUT)


# ------------------------------------------------------------
# Load JSONL
# ------------------------------------------------------------

records = []

with OUTPUT.open(
    "r",
    encoding="utf-8"
) as file:

    for line_number, line in enumerate(
        file,
        start=1
    ):

        line = line.strip()

        if not line:
            continue

        try:
            record = json.loads(line)

        except json.JSONDecodeError as exc:

            raise AssertionError(
                f"Invalid JSON at line {line_number}: {exc}"
            )

        records.append(record)


# ------------------------------------------------------------
# Record count
# ------------------------------------------------------------

print(
    "\nTotal records:",
    len(records)
)

assert len(records) == EXPECTED_RECORDS, (
    f"Expected {EXPECTED_RECORDS}, "
    f"got {len(records)}"
)

print(
    "Record count validation: PASS"
)


# ------------------------------------------------------------
# Schema validation
# ------------------------------------------------------------

required_keys = {
    "checkpoint",
    "decoded_sequences",
    "sequences_scores",
    "bugid",
    "hunk",
    "source",
    "target",
    "normalized_patch",
    "normalized_source",
    "normalized_target",
    "rank",
    "exact_match",
    "correct",
}

for index, record in enumerate(records):

    missing = (
        required_keys
        - set(record.keys())
    )

    assert not missing, (
        f"Record {index} missing keys: {missing}"
    )

print(
    "Schema validation: PASS"
)


# ============================================================
# CANDIDATE CONTENT VALIDATION
# ============================================================

non_empty_candidates = 0
empty_candidates = 0

for record in records:

    patch = str(
        record["decoded_sequences"]
    ).strip()

    if patch:
        non_empty_candidates += 1
    else:
        empty_candidates += 1


print("\nCandidate content:")
print(
    "  Total candidate records:",
    len(records)
)

print(
    "  Non-empty candidates:",
    non_empty_candidates
)

print(
    "  Empty candidates:",
    empty_candidates
)

assert non_empty_candidates > 0, (
    "No non-empty candidates found."
)

assert (
    non_empty_candidates + empty_candidates
    == len(records)
)

print(
    "Candidate content validation: PASS"
)


# ============================================================
# DISPLAY REAL GENERATED CANDIDATES
# ============================================================

print("\n" + "-" * 70)
print("GENERATED CANDIDATE SAMPLES")
print("-" * 70)

MAX_DISPLAY = 20
shown = 0

for record in records:

    patch = str(
        record["decoded_sequences"]
    ).strip()

    # Chỉ hiển thị candidate có patch thực tế
    if not patch:
        continue

    print(
        f"[{shown + 1}] "
        f"bugid={record['bugid']} | "
        f"hunk={record['hunk']} | "
        f"checkpoint={record['checkpoint']} | "
        f"rank={record['rank']} | "
        f"score={record['sequences_scores']} | "
        f"exact={record['exact_match']}"
    )

    print(
        f"    patch: {patch!r}"
    )

    shown += 1

    if shown >= MAX_DISPLAY:
        break


print(
    f"\nDisplayed {shown} non-empty candidate samples."
)

assert shown > 0, (
    "No candidate sample could be displayed."
)


# ============================================================
# CHECKPOINT DISTRIBUTION
# ============================================================

checkpoint_counts = Counter(
    record["checkpoint"]
    for record in records
)

print("\n" + "-" * 70)
print("CHECKPOINT DISTRIBUTION")
print("-" * 70)

for checkpoint, count in sorted(
    checkpoint_counts.items()
):

    print(
        f"  {checkpoint}: {count}"
    )


# ------------------------------------------------------------
# Check expected checkpoints
# ------------------------------------------------------------

actual_checkpoints = set(
    checkpoint_counts.keys()
)

missing_checkpoints = (
    EXPECTED_CHECKPOINTS
    - actual_checkpoints
)

assert not missing_checkpoints, (
    f"Missing checkpoints: {missing_checkpoints}"
)

print(
    "\nCheckpoint presence: PASS"
)


# ============================================================
# BUG / HUNK COVERAGE
# ============================================================

bugids = {
    record["bugid"]
    for record in records
}

bug_hunks = {
    (
        record["bugid"],
        record["hunk"]
    )
    for record in records
}

print("\n" + "-" * 70)
print("BUG / HUNK COVERAGE")
print("-" * 70)

print(
    "Unique bugids:",
    len(bugids)
)

print(
    "Bug/hunk combinations:",
    len(bug_hunks)
)

assert len(bugids) == EXPECTED_BUGIDS, (
    f"Expected {EXPECTED_BUGIDS} bugids, "
    f"got {len(bugids)}"
)

assert len(bug_hunks) == EXPECTED_BUG_HUNKS, (
    f"Expected {EXPECTED_BUG_HUNKS} bug/hunks, "
    f"got {len(bug_hunks)}"
)

print(
    "Bug coverage: PASS"
)

print(
    "Bug/hunk coverage: PASS"
)


# ============================================================
# EXACT MATCH
# ============================================================

exact_matches = sum(
    bool(record["exact_match"])
    for record in records
)

print(
    "\nExact-match candidates:",
    exact_matches
)


# ============================================================
# EMPTY PATCH
# ============================================================

empty_patches = sum(
    str(record["normalized_patch"]) == ""
    for record in records
)

print(
    "Empty patches:",
    empty_patches
)


# ============================================================
# REPRODUCIBILITY
# ============================================================

print("\n" + "-" * 70)
print("REPRODUCIBILITY METADATA")
print("-" * 70)

print(
    "Seed                :",
    EXPECTED_SEED
)

print(
    "Config hash         :",
    EXPECTED_CONFIG_HASH
)


# ============================================================
# FINAL SUMMARY
# ============================================================

print("\n" + "=" * 70)
print("STAGE 0 ARTIFACT SUMMARY")
print("=" * 70)

print(
    f"Raw candidates        : {EXPECTED_RAW_CANDIDATES:,}"
)

print(
    f"Final candidates      : {len(records):,}"
)

print(
    f"Deduplication         : "
    f"{EXPECTED_RAW_CANDIDATES:,} → {len(records):,}"
)

print(
    f"Non-empty candidates  : {non_empty_candidates:,}"
)

print(
    f"Empty candidates      : {empty_candidates:,}"
)

print(
    f"Unique bugids         : {len(bugids)}"
)

print(
    f"Bug/hunk combinations : {len(bug_hunks)}"
)

print(
    f"Exact-match candidates: {exact_matches}"
)


# ============================================================
# STAGE 0 SCOPE
# ============================================================

print("\n" + "=" * 70)
print("STAGE 0 SCOPE")
print("=" * 70)

print(
    "Artifact validation      : PASS"
)

print(
    "Candidate validation     : PASS"
)

print(
    "Candidate content check : PASS"
)

print(
    "Bug coverage             : PASS"
)

print(
    "Schema validation        : PASS"
)

print(
    "Pytest evaluation        : NOT PERFORMED"
)

print(
    "Full repair evaluation  : NOT PERFORMED"
)

print(
    "Repair rate             : NOT COMPUTED"
)

print(
    "Checkpoint ensemble     : NOT PERFORMED"
)


# ============================================================
# FINAL STATUS
# ============================================================

print("\n" + "=" * 70)
print("✅ STAGE 0 BASELINE INFERENCE: PASS")
print("=" * 70)

print(
    "Raw candidates       : 20,000"
)

print(
    "Final candidates     : 9,667"
)

print(
    "Non-empty candidates :",
    non_empty_candidates
)

print(
    "Bugids               : 40"
)

print(
    "Bug/hunks            : 40/40"
)

print(
    "Seed                 : 42"
)

print(
    "Config hash          : "
    "sha256:c0c516a4f019eabf3b648e39a901e7b81571636ba97dbc31ce8d64063962eaaa"
)

print(
    "\nSTATUS: READY FOR STAGE 1"
)

print("=" * 70)


In [ ]:
# ============================================================
# CELL 57 — QUIXBUGS-PYTHON PYTEST SMOKE TEST
# ============================================================
# Mục tiêu Stage 0:
#   1. Kiểm tra pytest đã được cài đặt.
#   2. Kiểm tra official QuixBugs-Python infrastructure.
#   3. Kiểm tra pytest có thể collect test.
#
# KHÔNG:
#   - đánh giá candidate patch
#   - chạy full pytest
#   - tính repair rate
#   - thực hiện checkpoint ensemble
#
# Lưu ý:
#   test_bitcount.py đã timeout khi execute.
#   Vì vậy smoke test này dùng --collect-only,
#   tức là chỉ collect test và KHÔNG execute test.
# ============================================================

import subprocess
import sys
from pathlib import Path

REPO = Path("/kaggle/working/MultiMend28")
QUIXBUGS_DIR = REPO / "benchmarks" / "QuixBugs"
TEST_DIR = QUIXBUGS_DIR / "python_testcases"

TIMEOUT_SECONDS = 60

print("=" * 70)
print("QUIXBUGS-PYTHON PYTEST SMOKE TEST")
print("=" * 70)

# ------------------------------------------------------------
# 1. Kiểm tra pytest
# ------------------------------------------------------------

pytest_version = subprocess.run(
    [
        sys.executable,
        "-m",
        "pytest",
        "--version",
    ],
    capture_output=True,
    text=True,
    timeout=TIMEOUT_SECONDS,
)

if pytest_version.returncode != 0:
    print("❌ pytest check FAILED")
    print(pytest_version.stderr)
    raise RuntimeError("pytest is not available.")

print("pytest:", pytest_version.stdout.strip())

# ------------------------------------------------------------
# 2. Kiểm tra official QuixBugs-Python files
# ------------------------------------------------------------

required_files = [
    QUIXBUGS_DIR / "conftest.py",
    QUIXBUGS_DIR / "tester.py",
    TEST_DIR / "load_testdata.py",
]

print("\nOfficial QuixBugs-Python infrastructure:")

for path in required_files:
    if path.is_file():
        print(f"  PASS: {path}")
    else:
        print(f"  FAIL: {path}")
        raise FileNotFoundError(
            f"Missing official QuixBugs-Python file: {path}"
        )

# ------------------------------------------------------------
# 3. Kiểm tra test files
# ------------------------------------------------------------

test_files = sorted(TEST_DIR.glob("test_*.py"))

print(f"\nQuixBugs-Python test files found: {len(test_files)}")

if not test_files:
    raise FileNotFoundError(
        f"No QuixBugs-Python test files found in {TEST_DIR}"
    )

print("Test discovery: PASS")

# ------------------------------------------------------------
# 4. Pytest collection-only
# ------------------------------------------------------------
# Quan trọng:
# --collect-only chỉ đọc/collect test.
# Không thực thi test nên không bị timeout bởi
# test_bitcount.py hoặc các test benchmark khác.

print("\nRunning pytest --collect-only...")

try:
    result = subprocess.run(
        [
            sys.executable,
            "-m",
            "pytest",
            "--collect-only",
            "-q",
            str(TEST_DIR),
        ],
        cwd=REPO,
        capture_output=True,
        text=True,
        timeout=TIMEOUT_SECONDS,
    )

except subprocess.TimeoutExpired:
    print("\n❌ PYTEST COLLECTION: TIMEOUT")
    print(f"Timeout: {TIMEOUT_SECONDS} seconds")
    raise

# ------------------------------------------------------------
# 5. Hiển thị kết quả collection
# ------------------------------------------------------------

if result.stdout.strip():
    print("\n---------- PYTEST COLLECTION ----------")
    print(result.stdout)

if result.stderr.strip():
    print("\n---------- PYTEST STDERR ----------")
    print(result.stderr)

print("\nPytest collection return code:", result.returncode)

# ------------------------------------------------------------
# 6. Kết luận Stage 0 smoke test
# ------------------------------------------------------------

if result.returncode == 0:

    print("\n" + "=" * 70)
    print("✅ QUIXBUGS-PYTHON PYTEST SMOKE TEST: PASS")
    print("=" * 70)

    print("pytest installation       : PASS")
    print("official infrastructure   : PASS")
    print("test discovery            : PASS")
    print("pytest collection         : PASS")

    print("\nIMPORTANT:")
    print("- No repair candidate was evaluated.")
    print("- No full pytest execution was performed.")
    print("- No checkpoint ensemble was performed.")
    print("- Full repair evaluation remains a later stage.")

else:

    print("\n" + "=" * 70)
    print("❌ QUIXBUGS-PYTHON PYTEST SMOKE TEST: FAIL")
    print("=" * 70)

    raise RuntimeError(
        "QuixBugs-Python pytest collection failed."
    )


In [ ]:
# ============================================================
# CELL 58 — STAGE 0 FINAL BASELINE REPORT
# ============================================================
# Chốt toàn bộ artifact Baseline Inference.
#
# Stage 0 KHÔNG:
#   - evaluate candidate patches
#   - chạy full pytest
#   - tính repair rate
#   - checkpoint ensemble
#
# Stage 1 sẽ sử dụng:
#   final_candidates_100.jsonl
# ============================================================

from pathlib import Path
import json


REPO = Path(
    "/kaggle/working/MultiMend28"
)

OUTPUT_DIR = (
    REPO
    / "outputs"
    / "QuixBugs-Python"
    / "outputs-multimend"
)

INPUT_FILE = (
    OUTPUT_DIR
    / "generated_input.jsonl"
)

RAW_FILE = (
    OUTPUT_DIR
    / "sequences_100.jsonl"
)

FINAL_FILE = (
    OUTPUT_DIR
    / "final_candidates_100.jsonl"
)

MANIFEST_FILE = (
    OUTPUT_DIR
    / "baseline_manifest.json"
)


print("=" * 70)
print("STAGE 0 — FINAL BASELINE REPORT")
print("=" * 70)


# ------------------------------------------------------------
# Artifact checks
# ------------------------------------------------------------

artifacts = {
    "generated_input.jsonl": INPUT_FILE,
    "sequences_100.jsonl": RAW_FILE,
    "final_candidates_100.jsonl": FINAL_FILE,
    "baseline_manifest.json": MANIFEST_FILE,
}

for name, path in artifacts.items():

    status = (
        "PASS"
        if path.exists()
        else "MISSING"
    )

    print(
        f"{name:<30} {status}"
    )

    assert path.exists(), (
        f"Missing artifact: {path}"
    )


# ------------------------------------------------------------
# Manifest
# ------------------------------------------------------------

with MANIFEST_FILE.open(
    "r",
    encoding="utf-8"
) as f:

    manifest = json.load(f)


print("\nManifest:")
print(
    json.dumps(
        manifest,
        indent=2,
        ensure_ascii=False,
    )
)


# ------------------------------------------------------------
# Final candidate count
# ------------------------------------------------------------

with FINAL_FILE.open(
    "r",
    encoding="utf-8"
) as f:

    final_count = sum(
        1
        for line in f
        if line.strip()
    )

print(
    "\nFinal candidates:",
    final_count
)

assert final_count == 9667


# ------------------------------------------------------------
# Stage 0 conclusion
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("✅ STAGE 0 BASELINE: COMPLETE")
print("=" * 70)

print("""
Dataset              : QuixBugs-Python
Model                : multimend-codet5-small
Seed                 : 42

Checkpoints          : 5
Checkpoint selection : trainer_state.global_step

Raw candidates       : 20,000
Combined candidates  : 9,667
Bugids               : 40
Bug/hunks            : 40

Input JSONL          : PASS
Raw output JSONL     : PASS
Final output JSONL   : PASS
Manifest             : PASS
Pytest smoke         : PASS

Repair evaluation    : NOT PERFORMED
Repair rate          : NOT COMPUTED
Checkpoint ensemble  : NOT PERFORMED

STATUS               : READY FOR STAGE 1
""")

print("=" * 70)


In [ ]:
# ================================================================
# # CELL 59 — STAGE 0 — BASELINE INFERENCE FINAL CHECK - CHẠY ALL LẠI ĐỂ KIỂM TRA ( KHÔNG CẦN )
# ================================================================
#
# Purpose:
#   1. Verify baseline inference artifacts.
#   2. Discover checkpoints from trainer_state.json.
#   3. Select the final 5 checkpoints by global_step.
#   4. Verify selected checkpoints against the baseline manifest.
#   5. Check official checkpoint availability if downloadable.
#   6. Validate config hash + seed + input/output JSONL.
#   7. Smoke-test official QuixBugs-Python infrastructure.
#
# Important:
#   - Checkpoint selection is based ONLY on trainer_state.global_step.
#   - Official checkpoint download is informative/non-blocking.
#   - Existing baseline artifacts are NOT regenerated here.
# ================================================================

from pathlib import Path
import json
import hashlib
import subprocess
import sys
import pandas as pd


# ================================================================
# 0. CONFIGURATION
# ================================================================

DATASET = "QuixBugs-Python"
MODEL = "multimend-codet5-small"
SEED = 42

CONFIG_HASH = (
    "sha256:"
    "c0c516a4f019eabf3b648e39a901e7b81571636ba97dbc31ce8d64063962eaaa"
)

BASE_DIR = Path("/kaggle/working/MultiMend28")

OUTPUT_DIR = (
    BASE_DIR
    / "outputs"
    / DATASET
    / "outputs-multimend"
)

MODEL_DIR = (
    BASE_DIR
    / "models"
    / MODEL
)

QUIXBUGS_DIR = (
    BASE_DIR
    / "benchmarks"
    / "QuixBugs"
)

# ------------------------------------------------
# Baseline artifacts
# ------------------------------------------------

INPUT_JSONL = OUTPUT_DIR / "generated_input.jsonl"
RAW_OUTPUT_JSONL = OUTPUT_DIR / "sequences_100.jsonl"
FINAL_OUTPUT_JSONL = OUTPUT_DIR / "final_candidates_100.jsonl"
MANIFEST_JSON = OUTPUT_DIR / "baseline_manifest.json"

# ------------------------------------------------
# Expected number of checkpoints
# ------------------------------------------------

N_CHECKPOINTS = 5

# ------------------------------------------------
# Expected baseline statistics
# ------------------------------------------------

EXPECTED_RAW_COUNT = 20_000
EXPECTED_FINAL_COUNT = 9_667
EXPECTED_BUG_COUNT = 40
EXPECTED_HUNK_COUNT = 40

# ------------------------------------------------
# Official QuixBugs infrastructure
# ------------------------------------------------

OFFICIAL_FILES = [
    QUIXBUGS_DIR / "conftest.py",
    QUIXBUGS_DIR / "tester.py",
    QUIXBUGS_DIR / "python_testcases" / "load_testdata.py",
]

# ================================================================
# 1. HELPERS
# ================================================================

def print_section(title):
    print()
    print("=" * 70)
    print(title)
    print("=" * 70)


def check_file(path, label=None):
    label = label or path.name

    ok = path.exists() and path.stat().st_size > 0

    print(
        f"{label:<40}: "
        f"{'PASS' if ok else 'FAIL'}"
    )

    if not ok:
        print(f"  -> Missing/empty: {path}")

    return ok


def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def sha256_file(path, chunk_size=1024 * 1024):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)

            if not chunk:
                break

            h.update(chunk)

    return h.hexdigest()


def read_jsonl(path):
    return pd.read_json(
        path,
        orient="records",
        lines=True,
    )


# ================================================================
# 2. INITIAL CONFIGURATION REPORT
# ================================================================

print_section("STAGE 0 — BASELINE CONFIGURATION")

print(f"Dataset              : {DATASET}")
print(f"Model                : {MODEL}")
print(f"Seed                 : {SEED}")
print(f"Config hash          : {CONFIG_HASH}")
print(f"Base directory       : {BASE_DIR}")
print(f"Output directory     : {OUTPUT_DIR}")
print(f"Model directory      : {MODEL_DIR}")
print(f"QuixBugs directory   : {QUIXBUGS_DIR}")

print()
print("Checkpoint policy:")
print("  Selection field    : trainer_state.global_step")
print("  Number selected    :", N_CHECKPOINTS)
print("  Ordering           : ascending global_step")
print("  Final checkpoints  : last 5")


# ================================================================
# 3. DISCOVER CHECKPOINTS FROM trainer_state.json
# ================================================================

print_section("CHECKPOINT DISCOVERY")

discovered_checkpoints = []

if MODEL_DIR.exists():
    checkpoint_dirs = sorted(
        MODEL_DIR.glob("checkpoint-*")
    )
else:
    checkpoint_dirs = []

print(f"Checkpoint directories found: {len(checkpoint_dirs)}")

for ckpt_dir in checkpoint_dirs:

    trainer_state = ckpt_dir / "trainer_state.json"

    if not trainer_state.exists():
        print(
            f"  {ckpt_dir.name:<30}: "
            "SKIP — trainer_state.json missing"
        )
        continue

    try:
        state = load_json(trainer_state)
        global_step = state.get("global_step")

        if global_step is None:
            print(
                f"  {ckpt_dir.name:<30}: "
                "SKIP — global_step missing"
            )
            continue

        discovered_checkpoints.append({
            "name": ckpt_dir.name,
            "path": str(ckpt_dir),
            "global_step": int(global_step),
            "trainer_state": str(trainer_state),
        })

        print(
            f"  {ckpt_dir.name:<30}: "
            f"global_step={int(global_step):,}"
        )

    except Exception as e:
        print(
            f"  {ckpt_dir.name:<30}: "
            f"FAIL — {type(e).__name__}: {e}"
        )


# ================================================================
# 4. SELECT FINAL 5 CHECKPOINTS
# ================================================================

print_section("CHECKPOINT SELECTION")

discovered_checkpoints = sorted(
    discovered_checkpoints,
    key=lambda x: x["global_step"],
)

if len(discovered_checkpoints) < N_CHECKPOINTS:
    selected_checkpoints = []
    checkpoint_selection_ok = False

    print(
        f"FAIL: only {len(discovered_checkpoints)} valid "
        f"checkpoints discovered; need {N_CHECKPOINTS}."
    )

else:
    selected_checkpoints = discovered_checkpoints[-N_CHECKPOINTS:]
    checkpoint_selection_ok = True

    print(
        f"Selected final {N_CHECKPOINTS} checkpoints "
        "using trainer_state.global_step:"
    )

    for rank, ckpt in enumerate(
        selected_checkpoints,
        start=1,
    ):
        print(
            f"  {rank}. "
            f"{ckpt['name']:<30} "
            f"global_step={ckpt['global_step']:,}"
        )


# ================================================================
# 5. EXPECTED CHECKPOINTS
# ================================================================

EXPECTED_CHECKPOINTS = [
    ("checkpoint-348600", 348600),
    ("checkpoint-406700", 406700),
    ("checkpoint-464800", 464800),
    ("checkpoint-522900", 522900),
    ("checkpoint-581008", 581008),
]

actual_checkpoints = [
    (
        x["name"],
        x["global_step"],
    )
    for x in selected_checkpoints
]

checkpoint_match_ok = (
    actual_checkpoints == EXPECTED_CHECKPOINTS
)

print()
print(
    f"{'Checkpoint selection validation':<40}: "
    f"{'PASS' if checkpoint_match_ok else 'FAIL'}"
)

if not checkpoint_match_ok:
    print()
    print("Expected:")
    for item in EXPECTED_CHECKPOINTS:
        print(" ", item)

    print("Actual:")
    for item in actual_checkpoints:
        print(" ", item)


# ================================================================
# 6. OFFICIAL CHECKPOINT AVAILABILITY
# ================================================================
#
# This section does NOT fail Stage 0 if an official checkpoint
# cannot be downloaded. It records availability for provenance.
#
# Replace OFFICIAL_CHECKPOINT_URL with the actual official URL
# when known.
# ================================================================

print_section("OFFICIAL CHECKPOINT CHECK")

OFFICIAL_CHECKPOINT_URL = None

official_checkpoint_status = "NOT_CONFIGURED"

if OFFICIAL_CHECKPOINT_URL:

    print("Official checkpoint URL configured:")
    print(f"  {OFFICIAL_CHECKPOINT_URL}")

    try:
        result = subprocess.run(
            [
                "python",
                "-c",
                (
                    "import urllib.request; "
                    f"urllib.request.urlopen('{OFFICIAL_CHECKPOINT_URL}', timeout=20)"
                ),
            ],
            capture_output=True,
            text=True,
            timeout=30,
        )

        if result.returncode == 0:
            official_checkpoint_status = "AVAILABLE"
        else:
            official_checkpoint_status = "UNAVAILABLE"

    except Exception as e:
        official_checkpoint_status = "UNAVAILABLE"

    print(
        f"{'Official checkpoint availability':<40}: "
        f"{official_checkpoint_status}"
    )

else:
    print(
        f"{'Official checkpoint availability':<40}: "
        "NOT CONFIGURED"
    )

print(
    "NOTE: official checkpoint availability is "
    "informational and does not invalidate Stage 0."
)


# ================================================================
# 7. BASELINE ARTIFACT CHECK
# ================================================================

print_section("BASELINE ARTIFACT VALIDATION")

input_ok = check_file(
    INPUT_JSONL,
    "generated_input.jsonl",
)

raw_ok = check_file(
    RAW_OUTPUT_JSONL,
    "sequences_100.jsonl",
)

final_ok = check_file(
    FINAL_OUTPUT_JSONL,
    "final_candidates_100.jsonl",
)

manifest_ok = check_file(
    MANIFEST_JSON,
    "baseline_manifest.json",
)


# ================================================================
# 8. MANIFEST VALIDATION
# ================================================================

print_section("MANIFEST VALIDATION")

manifest = None
manifest_ok_content = False

if MANIFEST_JSON.exists():

    try:
        manifest = load_json(MANIFEST_JSON)

        cfg = manifest.get("config", {})

        checks = {
            "dataset": cfg.get("dataset") == DATASET,
            "model_name": cfg.get("model_name") == MODEL,
            "seed": cfg.get("seed") == SEED,
            "config_hash": cfg.get("config_hash") == CONFIG_HASH,
            "num_checkpoints": cfg.get("num_checkpoints")
            == N_CHECKPOINTS,
        }

        for key, ok in checks.items():
            print(
                f"{key:<40}: "
                f"{'PASS' if ok else 'FAIL'}"
            )

        manifest_ok_content = all(checks.values())

    except Exception as e:
        print(
            f"{'Manifest JSON parsing':<40}: FAIL"
        )
        print(f"  -> {type(e).__name__}: {e}")

else:
    print(
        f"{'Manifest configuration':<40}: FAIL"
    )


# ================================================================
# 9. MANIFEST CHECKPOINT VALIDATION
# ================================================================

manifest_checkpoint_ok = False

if manifest is not None:

    selected = manifest.get(
        "selected_checkpoints",
        [],
    )

    manifest_actual = [
        (
            x.get("name"),
            x.get("global_step"),
        )
        for x in selected
    ]

    manifest_checkpoint_ok = (
        manifest_actual == EXPECTED_CHECKPOINTS
    )

print(
    f"{'Manifest checkpoint selection':<40}: "
    f"{'PASS' if manifest_checkpoint_ok else 'FAIL'}"
)


# ================================================================
# 10. INPUT JSONL VALIDATION
# ================================================================

print_section("INPUT JSONL VALIDATION")

input_count = None
input_columns = []

if INPUT_JSONL.exists():

    try:
        input_df = read_jsonl(INPUT_JSONL)

        input_count = len(input_df)
        input_columns = list(input_df.columns)

        print(
            f"{'Input records':<40}: "
            f"{input_count:,}"
        )

        print(
            f"{'Input columns':<40}: "
            f"{', '.join(input_columns)}"
        )

        print(
            f"{'Input JSONL parsing':<40}: PASS"
        )

    except Exception as e:
        print(
            f"{'Input JSONL parsing':<40}: FAIL"
        )
        print(f"  -> {type(e).__name__}: {e}")


# ================================================================
# 11. RAW OUTPUT VALIDATION
# ================================================================

print_section("RAW BASELINE OUTPUT VALIDATION")

raw_count = None
checkpoint_counts = {}

if RAW_OUTPUT_JSONL.exists():

    try:

        raw_df = read_jsonl(RAW_OUTPUT_JSONL)

        raw_count = len(raw_df)

        print(
            f"{'Raw candidates':<40}: "
            f"{raw_count:,}"
        )

        if "checkpoint" in raw_df.columns:

            checkpoint_counts = (
                raw_df["checkpoint"]
                .value_counts()
                .to_dict()
            )

            print()
            print("Candidates per checkpoint:")

            for name, step in EXPECTED_CHECKPOINTS:

                count = checkpoint_counts.get(
                    name,
                    0,
                )

                print(
                    f"  {name:<30}: "
                    f"{count:,}"
                )

        raw_count_ok = (
            raw_count == EXPECTED_RAW_COUNT
        )

        print(
            f"{'Raw candidate count':<40}: "
            f"{'PASS' if raw_count_ok else 'FAIL'}"
        )

    except Exception as e:

        raw_count_ok = False

        print(
            f"{'Raw JSONL parsing':<40}: FAIL"
        )

        print(
            f"  -> {type(e).__name__}: {e}"
        )

else:
    raw_count_ok = False


# ================================================================
# 12. FINAL CANDIDATE VALIDATION
# ================================================================

print_section("FINAL CANDIDATE VALIDATION")

final_count = None
bug_count = None
hunk_count = None

if FINAL_OUTPUT_JSONL.exists():

    try:

        final_df = read_jsonl(
            FINAL_OUTPUT_JSONL
        )

        final_count = len(final_df)

        if "bugid" in final_df.columns:
            bug_count = (
                final_df["bugid"]
                .nunique()
            )

        if {
            "bugid",
            "hunk",
        }.issubset(final_df.columns):

            hunk_count = (
                final_df[
                    ["bugid", "hunk"]
                ]
                .drop_duplicates()
                .shape[0]
            )

        print(
            f"{'Final candidates':<40}: "
            f"{final_count:,}"
        )

        print(
            f"{'Unique bugids':<40}: "
            f"{bug_count}"
        )

        print(
            f"{'Bug/hunk combinations':<40}: "
            f"{hunk_count}"
        )

    except Exception as e:

        print(
            f"{'Final JSONL parsing':<40}: FAIL"
        )

        print(
            f"  -> {type(e).__name__}: {e}"
        )


final_count_ok = (
    final_count == EXPECTED_FINAL_COUNT
)

bug_count_ok = (
    bug_count == EXPECTED_BUG_COUNT
)

hunk_count_ok = (
    hunk_count == EXPECTED_HUNK_COUNT
)

print()
print(
    f"{'Final candidate count':<40}: "
    f"{'PASS' if final_count_ok else 'FAIL'}"
)

print(
    f"{'Unique bugid count':<40}: "
    f"{'PASS' if bug_count_ok else 'FAIL'}"
)

print(
    f"{'Bug/hunk count':<40}: "
    f"{'PASS' if hunk_count_ok else 'FAIL'}"
)


# ================================================================
# 13. QUIXBUGS OFFICIAL INFRASTRUCTURE SMOKE
# ================================================================

print_section("QUIXBUGS-PYTHON OFFICIAL SMOKE")

official_files_status = {}

for path in OFFICIAL_FILES:

    ok = path.exists()

    official_files_status[path.name] = ok

    print(
        f"{path.name:<40}: "
        f"{'PASS' if ok else 'FAIL'}"
    )

official_ok = all(
    official_files_status.values()
)

# ------------------------------------------------
# Discover test files
# ------------------------------------------------

test_dir = (
    QUIXBUGS_DIR
    / "python_testcases"
)

test_files = sorted(
    test_dir.glob("test_*.py")
) if test_dir.exists() else []

print()
print(
    f"{'Test files discovered':<40}: "
    f"{len(test_files)}"
)

test_file_count_ok = (
    len(test_files) == 40
)

print(
    f"{'Expected 40 test files':<40}: "
    f"{'PASS' if test_file_count_ok else 'FAIL'}"
)


# ================================================================
# 14. PYTEST COLLECTION
# ================================================================

print_section("PYTEST COLLECTION")

pytest_collection_ok = False
pytest_collected = None
pytest_output = ""

if test_dir.exists():

    try:

        result = subprocess.run(
            [
                sys.executable,
                "-m",
                "pytest",
                str(test_dir),
                "--collect-only",
                "-q",
            ],
            cwd=str(QUIXBUGS_DIR),
            capture_output=True,
            text=True,
            timeout=300,
        )

        stdout = result.stdout or ""
        stderr = result.stderr or ""

        pytest_output = stdout + "\n" + stderr

        # --------------------------------------------------------
        # IMPORTANT:
        # Pytest may report either:
        #
        #   278 tests collected in 0.08s
        #
        # or:
        #
        #   278 items collected
        #
        # Support both formats.
        # --------------------------------------------------------

        import re

        collection_patterns = [
            r"(\d+)\s+tests?\s+collected",
            r"(\d+)\s+items?\s+collected",
            r"collected\s+(\d+)\s+tests?",
            r"collected\s+(\d+)\s+items?",
        ]

        for pattern in collection_patterns:

            match = re.search(
                pattern,
                pytest_output,
                flags=re.IGNORECASE,
            )

            if match:
                pytest_collected = int(
                    match.group(1)
                )
                break

        # --------------------------------------------------------
        # Do NOT print every collected test unless debugging.
        # This keeps the Stage 0 report compact.
        # --------------------------------------------------------

        print(
            f"Pytest return code                 : "
            f"{result.returncode}"
        )

        print(
            f"Tests collected                    : "
            f"{pytest_collected if pytest_collected is not None else 'N/A'}"
        )

        pytest_collection_ok = (
            result.returncode == 0
            and pytest_collected == 278
        )

        print(
            f"{'Pytest collection':<40}: "
            f"{'PASS' if pytest_collection_ok else 'FAIL'}"
        )

        # --------------------------------------------------------
        # If parsing failed, show pytest's final summary
        # instead of dumping hundreds of test names.
        # --------------------------------------------------------

        if pytest_collected is None:

            print()
            print("Pytest output tail:")
            print(
                "\n".join(
                    pytest_output.strip().splitlines()[-10:]
                )
            )

    except subprocess.TimeoutExpired:

        pytest_collection_ok = False

        print(
            f"{'Pytest collection':<40}: FAIL"
        )

        print(
            "Reason: pytest collection timed out."
        )

    except Exception as e:

        pytest_collection_ok = False

        print(
            f"{'Pytest collection':<40}: FAIL"
        )

        print(
            f"Reason: {type(e).__name__}: {e}"
        )

else:

    print(
        f"{'Pytest collection':<40}: FAIL"
    )

    print(
        "Reason: QuixBugs python_testcases directory missing."
    )

# ================================================================
# 15. BASELINE HASH / PROVENANCE REPORT
# ================================================================

print_section("BASELINE PROVENANCE")

print(f"Dataset              : {DATASET}")
print(f"Model                : {MODEL}")
print(f"Seed                 : {SEED}")
print(f"Config hash          : {CONFIG_HASH}")
print(
    "Checkpoint selection : "
    "trainer_state.global_step"
)
print(f"Checkpoints           : {N_CHECKPOINTS}")

print()
print("Artifacts:")

for path in [
    INPUT_JSONL,
    RAW_OUTPUT_JSONL,
    FINAL_OUTPUT_JSONL,
    MANIFEST_JSON,
]:

    if path.exists():

        print(
            f"  {path.name:<32} "
            f"size={path.stat().st_size:,} bytes"
        )

    else:

        print(
            f"  {path.name:<32} "
            "MISSING"
        )

# ================================================================
# 16. PYTEST COLLECTION
# ================================================================

print_section("PYTEST COLLECTION")

pytest_collection_ok = False
pytest_collected = None
pytest_exit_code = None
pytest_collection_error = None

EXPECTED_TEST_FILES = 40
EXPECTED_TESTS_COLLECTED = 278

try:
    import pytest

    # ------------------------------------------------------------
    # Custom plugin to capture collection statistics
    # ------------------------------------------------------------
    class CollectionStatsPlugin:

        def __init__(self):
            self.tests_collected = 0
            self.collection_errors = 0
            self.session_started = False
            self.session_finished = False

        def pytest_sessionstart(self, session):
            self.session_started = True

        def pytest_collection_finish(self, session):
            self.tests_collected = len(session.items)
            self.collection_errors = len(session.testsfailed)

        def pytest_sessionfinish(self, session, exitstatus):
            self.session_finished = True

    collection_plugin = CollectionStatsPlugin()

    # ------------------------------------------------------------
    # Run collection only
    #
    # --collect-only:
    #   Collect tests without executing them.
    #
    # -q:
    #   Keep output compact.
    # ------------------------------------------------------------
    pytest_args = [
        str(QUIXBUGS_DIR / "python_testcases"),
        "--collect-only",
        "-q",
    ]

    pytest_exit_code = pytest.main(
        pytest_args,
        plugins=[collection_plugin],
    )

    pytest_collected = collection_plugin.tests_collected

    # ------------------------------------------------------------
    # Collection is considered successful when:
    #
    # 1. pytest actually started
    # 2. pytest finished collection
    # 3. no collection error occurred
    # 4. expected number of tests was collected
    #
    # NOTE:
    # We intentionally do NOT require a full pytest run here.
    # Stage 0 only validates the official test infrastructure
    # and collection.
    # ------------------------------------------------------------
    pytest_collection_ok = (
        collection_plugin.session_started
        and collection_plugin.session_finished
        and collection_plugin.collection_errors == 0
        and pytest_collected == EXPECTED_TESTS_COLLECTED
    )

except Exception as exc:

    pytest_collection_error = repr(exc)
    pytest_collection_ok = False

# ------------------------------------------------------------
# Collection report
# ------------------------------------------------------------
print(
    f"{'Expected tests':<40}: "
    f"{EXPECTED_TESTS_COLLECTED}"
)

print(
    f"{'Tests collected':<40}: "
    f"{pytest_collected if pytest_collected is not None else 'N/A'}"
)

print(
    f"{'Pytest exit code':<40}: "
    f"{pytest_exit_code if pytest_exit_code is not None else 'N/A'}"
)

print(
    f"{'Collection errors':<40}: "
    f"{collection_plugin.collection_errors}"
    if 'collection_plugin' in locals()
    else "Collection errors".ljust(40) + ": N/A"
)

print(
    f"{'Pytest collection':<40}: "
    f"{'PASS' if pytest_collection_ok else 'FAIL'}"
)

if pytest_collection_error is not None:
    print(
        f"{'Collection error':<40}: "
        f"{pytest_collection_error}"
    )


# ================================================================
# 17. FINAL STAGE 0 CHECKS
# ================================================================

# ------------------------------------------------------------
# Keep official checkpoint informational.
#
# "NOT_CONFIGURED" must NOT invalidate Stage 0 because the
# experiment can legitimately proceed with the locally available
# checkpoints.
# ------------------------------------------------------------
official_checkpoint_ok = True


# ------------------------------------------------------------
# Build final check dictionary
# ------------------------------------------------------------
stage0_checks = {
    "generated_input.jsonl": input_ok,
    "sequences_100.jsonl": raw_ok,
    "final_candidates_100.jsonl": final_ok,
    "baseline_manifest.json": manifest_ok,

    "manifest configuration": manifest_ok_content,

    "checkpoint discovery": (
        len(discovered_checkpoints) >= 5
        if "discovered_checkpoints" in locals()
        else len(selected_checkpoints) >= 5
    ),

    "checkpoint selection": checkpoint_ok,

    "manifest checkpoints": checkpoint_ok,

    "raw candidate count": (
        raw_count == 20_000
    ),

    "final candidate count": (
        final_count == 9_667
    ),

    "unique bugid count": (
        bug_count == 40
    ),

    "bug/hunk count": (
        hunk_count == 40
    ),

    "official QuixBugs infrastructure": (
        official_ok
    ),

    "40 test files": (
        len(test_files) == EXPECTED_TEST_FILES
    ),

    "pytest collection": (
        pytest_collection_ok
    ),
}


# ------------------------------------------------------------
# Final Stage 0 decision
#
# Official checkpoint availability is intentionally excluded
# from the blocking checks.
# ------------------------------------------------------------
stage0_pass = all(stage0_checks.values())


# ================================================================
# 18. FINAL SUMMARY TABLE
# ================================================================

print_section("STAGE 0 — FINAL SUMMARY")

for name, ok in stage0_checks.items():

    print(
        f"{name:<40}: "
        f"{'PASS' if ok else 'FAIL'}"
    )

print()
print("-" * 70)

if stage0_pass:
    print("✅ STAGE 0 BASELINE: COMPLETE")
else:
    print("❌ STAGE 0 BASELINE: INCOMPLETE")

print("-" * 70)

print()
print("Dataset              :", DATASET)
print("Model                :", MODEL)
print("Seed                 :", SEED)
print("Config hash          :", CONFIG_HASH)

print(
    "Checkpoint selection : "
    "trainer_state.global_step"
)

print(
    "Checkpoints          : "
    f"{len(selected_checkpoints)}"
)

print(
    "Raw candidates       : "
    f"{raw_count:,}"
    if raw_count is not None
    else "Raw candidates       : N/A"
)

print(
    "Final candidates     : "
    f"{final_count:,}"
    if final_count is not None
    else "Final candidates     : N/A"
)

print(
    "Bugids               : "
    f"{bug_count}"
    if bug_count is not None
    else "Bugids               : N/A"
)

print(
    "Bug/hunks            : "
    f"{hunk_count}"
    if hunk_count is not None
    else "Bug/hunks            : N/A"
)

print(
    "Pytest smoke         : "
    f"{'PASS' if pytest_collection_ok else 'FAIL'}"
)

print(
    "Tests collected      : "
    f"{pytest_collected}"
    if pytest_collected is not None
    else "Tests collected      : N/A"
)

print(
    "Official checkpoint  : "
    f"{official_checkpoint_status}"
)

print()
print("Repair candidate evaluation : NOT PERFORMED")
print("Full pytest execution       : NOT PERFORMED")
print("Checkpoint ensemble         : NOT PERFORMED")

print()

if stage0_pass:
    print("STATUS                : READY FOR STAGE 1")
else:
    print("STATUS                : BLOCKED — FIX FAILED CHECKS")

print("=" * 70)


# ================================================================
# 19. MACHINE-READABLE RESULT
# ================================================================

stage0_result = {
    "stage": 0,

    "status": (
        "READY_FOR_STAGE_1"
        if stage0_pass
        else "BLOCKED"
    ),

    "dataset": DATASET,
    "model": MODEL,
    "seed": SEED,
    "config_hash": CONFIG_HASH,

    "checkpoint_selection": {
        "field": "trainer_state.global_step",
        "num_checkpoints": len(selected_checkpoints),
        "ordering": "ascending_global_step",
        "policy": "last_5",
    },

    "selected_checkpoints": [
        {
            "name": x["name"],
            "global_step": x["global_step"],
        }
        for x in selected_checkpoints
    ],

    "artifacts": {
        "generated_input": {
            "path": str(INPUT_JSONL),
            "exists": INPUT_JSONL.exists(),
            "size_bytes": (
                INPUT_JSONL.stat().st_size
                if INPUT_JSONL.exists()
                else None
            ),
            "valid": input_ok,
        },

        "raw_output": {
            "path": str(RAW_OUTPUT_JSONL),
            "exists": RAW_OUTPUT_JSONL.exists(),
            "size_bytes": (
                RAW_OUTPUT_JSONL.stat().st_size
                if RAW_OUTPUT_JSONL.exists()
                else None
            ),
            "valid": raw_ok,
        },

        "final_candidates": {
            "path": str(FINAL_OUTPUT_JSONL),
            "exists": FINAL_OUTPUT_JSONL.exists(),
            "size_bytes": (
                FINAL_OUTPUT_JSONL.stat().st_size
                if FINAL_OUTPUT_JSONL.exists()
                else None
            ),
            "valid": final_ok,
        },

        "baseline_manifest": {
            "path": str(MANIFEST_JSON),
            "exists": MANIFEST_JSON.exists(),
            "size_bytes": (
                MANIFEST_JSON.stat().st_size
                if MANIFEST_JSON.exists()
                else None
            ),
            "valid": manifest_ok,
        },
    },

    "input": {
        "records": (
            len(input_df)
            if "input_df" in locals() and input_df is not None
            else None
        ),
        "columns": (
            list(input_df.columns)
            if "input_df" in locals() and input_df is not None
            else None
        ),
        "jsonl_valid": input_ok,
    },

    "raw_baseline": {
        "candidates": raw_count,
        "expected_candidates": 20_000,

        "candidates_per_checkpoint": {
            name: checkpoint_counts.get(name, 0)
            for name, _ in EXPECTED_CHECKPOINTS
        },

        "count_valid": raw_count == 20_000,
    },

    "final_candidates": {
        "candidates": final_count,
        "expected_candidates": 9_667,

        "unique_bugids": bug_count,
        "expected_bugids": 40,

        "bug_hunk_combinations": hunk_count,
        "expected_bug_hunk_combinations": 40,
    },

    "quixbugs": {
        "official_infrastructure": official_ok,
        "test_files": len(test_files),
        "expected_test_files": EXPECTED_TEST_FILES,

        "pytest_collection": {
            "status": (
                "PASS"
                if pytest_collection_ok
                else "FAIL"
            ),
            "tests_collected": pytest_collected,
            "expected_tests": EXPECTED_TESTS_COLLECTED,
            "exit_code": pytest_exit_code,
            "collection_errors": (
                collection_plugin.collection_errors
                if "collection_plugin" in locals()
                else None
            ),
        },
    },

    "official_checkpoint": {
        "status": official_checkpoint_status,
        "informational_only": True,
    },

    "stage1": {
        "repair_candidate_evaluation": "NOT_PERFORMED",
        "full_pytest": "NOT_PERFORMED",
        "checkpoint_ensemble": "NOT_PERFORMED",
    },

    "checks": stage0_checks,
}


# ================================================================
# 20. SAVE MACHINE-READABLE RESULT
# ================================================================

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

STAGE0_RESULT_JSON = OUTPUT_DIR / "stage0_final_check.json"

with open(
    STAGE0_RESULT_JSON,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        stage0_result,
        f,
        indent=2,
        ensure_ascii=False,
    )

print()
print(
    "Machine-readable result saved to:"
)

print(
    f"  {STAGE0_RESULT_JSON}"
)


**Bư****ớc khởi đầu Train model của Papper******

In [ ]:
# Cell 1 — Kaggle environment
import os, sys, subprocess, textwrap, json, pathlib, shutil, time
from pathlib import Path

print("Python:", sys.version)
print("CUDA_VISIBLE_DEVICES:", os.environ.get("CUDA_VISIBLE_DEVICES"))
try:
    import torch
    print("Torch:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
except Exception as e:
    print("Torch import error:", repr(e))


In [ ]:
# Cell 2 — Clone ONLY the MultiMend repo first.
# We intentionally do not recurse through all benchmark submodules yet.

REPO = Path("/kaggle/working/MultiMend")
if REPO.exists():
    print("Repo already exists:", REPO)
else:
    subprocess.run([
        "git", "clone", "--depth", "1",
        "https://github.com/h4iku/MultiMend.git",
        str(REPO)
    ], check=True)

os.chdir(REPO)
print("HEAD:", subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip())
print("Repo:", REPO)
print("Top-level:", [p.name for p in REPO.iterdir()])


In [ ]:
# Cell 3 — Inspect the official repo BEFORE running anything
from pathlib import Path

for p in sorted((REPO / "src").rglob("*.py")):
    print(p.relative_to(REPO))

print("\nrequirements.txt:")
print((REPO / "requirements.txt").read_text())


## Important implementation findings

The official generation script uses:

- `max_input_length = 512`
- `max_target_length = 256`
- `beam_size = 100`
- `num_return_sequences = 100`
- `batch_size = 1`
- `num_checkpoints = 5`

It loads the last five `checkpoint-*` directories under `models/multimend-codet5-small`, generates from every checkpoint, and writes `sequences_100.jsonl`.

The context augmentation is local RAG: `all-MiniLM-L6-v2` + persistent ChromaDB, not a paid LLM API.


In [ ]:
# Cell 4 — Install official MultiMend dependencies.
# Do NOT upgrade unrelated Kaggle packages.

import subprocess
import sys

result = subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "--no-cache-dir",
        "-r",
        "requirements.txt",
        "more-itertools",
        "pygments",
    ],
    text=True,
)

if result.returncode != 0:
    raise RuntimeError("MultiMend dependency installation failed.")

print("MultiMend dependency installation complete.")

In [ ]:
# Cell 4b — Force the versions required by MultiMend
import subprocess
import sys

packages = [
    "numpy==1.26.4",
    "pandas==2.2.3",
    "torch==2.4.0",
]

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "--no-cache-dir",
        "--force-reinstall",
        *packages,
    ],
    check=True,
)

print("Pinned NumPy/Pandas/Torch versions installed.")

In [ ]:
# Cell 4c — Match torchvision with MultiMend's torch 2.4.0

import subprocess
import sys

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "--no-cache-dir",
        "--force-reinstall",
        "torchvision==0.19.0",
    ],
    check=True,
)

print("torchvision 0.19.0 installed.")

In [ ]:
# Fix NumPy for ChromaDB 0.5.5 / MultiMend
import subprocess
import sys

subprocess.run(
    [
        sys.executable,
        "-m", "pip", "install",
        "-q",
        "--no-cache-dir",
        "--force-reinstall",
        "numpy==1.26.4",
    ],
    check=True,
)

print("NumPy 1.26.4 installed.")


In [ ]:
import numpy as np

print("NumPy:", np.__version__)


In [ ]:
import torch
import transformers
import datasets
import chromadb
import sentence_transformers
import tree_sitter
import unidiff
import more_itertools
import pygments

print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("datasets:", datasets.__version__)
print("chromadb:", chromadb.__version__)
print("sentence-transformers:", sentence_transformers.__version__)
print("tree-sitter:", tree_sitter.__version__ if hasattr(tree_sitter, "__version__") else "unknown")
print("CUDA:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
# Cell 5 — Verify versions and direct imports
import torch, transformers, datasets, chromadb, sentence_transformers
import tree_sitter, unidiff, more_itertools, pygments

print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("datasets:", datasets.__version__)
print("chromadb:", chromadb.__version__)
print("sentence-transformers:", sentence_transformers.__version__)
print("tree-sitter:", tree_sitter.__version__ if hasattr(tree_sitter, "__version__") else "unknown")
print("CUDA:", torch.cuda.is_available())


In [ ]:
from pathlib import Path

REPO = Path("/kaggle/working/MultiMend")

print("Repo exists:", REPO.exists())
print("Repo:", REPO)

if REPO.exists():
    print("Contents:")
    for p in list(REPO.iterdir())[:20]:
        print(" -", p.name)


In [ ]:
# Cell 6 — Download QuixBugs

from pathlib import Path
import subprocess
import shutil

REPO = Path("/kaggle/working/MultiMend")
QB = REPO / "benchmarks" / "QuixBugs"

if not REPO.exists():
    raise FileNotFoundError(
        f"MultiMend repo not found at {REPO}. "
        "Please run the clone cell first."
    )

if not (QB / "python_programs").exists():
    if QB.exists():
        shutil.rmtree(QB)

    subprocess.run([
        "git", "clone", "--depth", "1",
        "https://github.com/jkoppel/QuixBugs.git",
        str(QB)
    ], check=True)

print("QuixBugs files:")
for d in ["python_programs", "correct_python_programs", "python_testcases"]:
    p = QB / d
    print(
        d,
        "=>", p.exists(),
        "count=", len(list(p.glob("*.py"))) if p.exists() else 0
    )


In [ ]:
# Check what MultiMend itself actually references
!grep -RniE "from_pretrained|snapshot_download|huggingface|checkpoint" \
    /kaggle/working/MultiMend/src \
    --include="*.py" | head -200


xác định chính xác workflow/config của tác giả

In [ ]:
# Cell 7a — Inspect MultiMend model/config paths

print("=== configs.py ===")
print((REPO / "src" / "configs.py").read_text())

print("\n=== generate_primary_candidates.py (relevant section) ===")
lines = (REPO / "src" / "generate_primary_candidates.py").read_text().splitlines()
for i, line in enumerate(lines):
    if 180 <= i <= 235:
        print(f"{i+1}: {line}")


In [ ]:
# Cell 7b — Inspect existing model/results artifacts

from pathlib import Path

print("=== models/ ===")
models_dir = REPO / "models"

if models_dir.exists():
    for p in models_dir.rglob("*"):
        if p.is_file():
            print(p.relative_to(REPO))
else:
    print("No models directory yet.")

print("\n=== results/ ===")
results_dir = REPO / "results"

for p in results_dir.rglob("*"):
    if p.is_file():
        print(p.relative_to(REPO))


In [ ]:
# # Cell 7 — Download the OFFICIAL MultiMend fine-tuned checkpoint repository.
# # No API key is required for a public Hugging Face repository.

# from huggingface_hub import snapshot_download

# MODEL_REPO = "h4iku/multimend-codet5-small"
# MODEL_DIR = REPO / "models" / "multimend-codet5-small"
# MODEL_DIR.mkdir(parents=True, exist_ok=True)

# snapshot_download(
#     repo_id=MODEL_REPO,
#     local_dir=str(MODEL_DIR),
#     local_dir_use_symlinks=False,
# )

# print("Model snapshot:", MODEL_DIR)
# print("Checkpoint directories:")
# for p in sorted(MODEL_DIR.glob("checkpoint-*")):
#     print(" ", p.name)


In [ ]:
# Cell 8 — Inspect the official MultiMend training configuration

train_file = REPO / "src" / "train_model.py"
lines = train_file.read_text().splitlines()

for i, line in enumerate(lines, start=1):
    print(f"{i:3}: {line}")


tính chính xác số sample + số training steps + save_steps từ chính 4 dataset mà code dùng. (Chạy CPU thôi, không tốn GPU)

In [ ]:
# Cell 9 — Inspect official CoCoNuT training dataset sizes

from datasets import load_dataset

dataset_names = {
    "Python": "h4iku/coconut_python2010_preprocessed",
    "Java": "h4iku/coconut_java2006_preprocessed",
    "JavaScript": "h4iku/coconut_javascript2010_preprocessed",
    "C": "h4iku/coconut_c2005_preprocessed",
}

for prefix, name in dataset_names.items():
    print(f"\nLoading {prefix}: {name}")
    ds = load_dataset(name, split="train")
    print("raw samples:", len(ds))
    print("columns:", ds.column_names)


đã biết raw dataset tổng cộng khoảng 2.64 triệu samples, nhưng code còn filter rất nhiều sample. Cần tính số sample sau filter, vì đó mới quyết định 30 GPU-hours có đủ không.

In [ ]:
# Cell 10 — Reproduce the official filtering step and calculate training steps

from math import ceil
from datasets import load_dataset, concatenate_datasets
from transformers import AutoTokenizer

model_checkpoint = "Salesforce/codet5-small"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

dataset_names = {
    "Python": "h4iku/coconut_python2010_preprocessed",
    "Java": "h4iku/coconut_java2006_preprocessed",
    "JavaScript": "h4iku/coconut_javascript2010_preprocessed",
    "C": "h4iku/coconut_c2005_preprocessed",
}

max_input_length = 512
max_target_length = 256


def preprocess_function(prefix, examples):
    inputs = [
        f"{prefix} {ex.strip()} :".replace(
            tokenizer.eos_token, tokenizer.unk_token
        )
        for ex in examples["rem"]
    ]

    targets = [
        ex.strip().replace(
            tokenizer.eos_token, tokenizer.unk_token
        )
        for ex in examples["add"]
    ]

    contexts = [
        " ".join(ex.split()).replace(
            tokenizer.eos_token, tokenizer.unk_token
        )
        for ex in examples["context"]
    ]

    inputs_contexts = [
        f"{src} {ctx}" for src, ctx in zip(inputs, contexts)
    ]

    model_inputs = tokenizer(
        inputs_contexts,
        max_length=max_input_length,
        truncation=True,
    )

    model_inputs["inputs_only"] = tokenizer(inputs).input_ids

    labels = tokenizer(targets)
    model_inputs["labels"] = labels["input_ids"]

    return model_inputs


def allowed_length(example):
    return (
        len(example["inputs_only"]) <= max_input_length
        and 2 < len(example["labels"]) <= max_target_length
    )


filtered_datasets = []

for prefix, dataset_name in dataset_names.items():
    print(f"\nProcessing {prefix}...")

    raw_dataset = load_dataset(dataset_name, split="train")

    tokenized = raw_dataset.map(
        lambda ex: preprocess_function(prefix, ex),
        batched=True,
        remove_columns=raw_dataset.column_names,
        desc=f"Tokenizing {prefix}",
    )

    filtered = tokenized.filter(
        allowed_length,
        desc=f"Filtering {prefix}",
    ).remove_columns(["inputs_only"])

    print("Raw:", len(raw_dataset))
    print("After filter:", len(filtered))

    filtered_datasets.append(filtered)


concatenated_dataset = concatenate_datasets(filtered_datasets)

print("\n==============================")
print("FINAL DATASET")
print("==============================")
print("Total training samples:", len(concatenated_dataset))

train_batch_size = 8
epochs = 2

epoch_steps = ceil(len(concatenated_dataset) / train_batch_size)
train_steps = epoch_steps * epochs
save_steps = epoch_steps // 5

print("Steps / epoch:", epoch_steps)
print("Total training steps:", train_steps)
print("Checkpoint save_steps:", save_steps)
print("Expected checkpoints:", 10)


In [ ]:
# Cell 11 — Benchmark official MultiMend training speed
# Uses the exact official model/training batch size.
# Only 200 steps, then estimate full training time.

import time
import torch
from transformers import (
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
)

model_checkpoint = "Salesforce/codet5-small"

model = AutoModelForSeq2SeqLM.from_pretrained(model_checkpoint)

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
)

benchmark_args = Seq2SeqTrainingArguments(
    output_dir=str(REPO / "benchmark_train"),
    per_device_train_batch_size=8,
    max_steps=200,
    fp16=True,
    learning_rate=1e-4,
    lr_scheduler_type="constant",
    logging_steps=50,
    save_strategy="no",
    report_to="none",
    remove_unused_columns=False,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=benchmark_args,
    train_dataset=concatenated_dataset,
    data_collator=data_collator,
    tokenizer=tokenizer,
)

print("Starting 200-step benchmark...")
start = time.time()

trainer.train()

elapsed = time.time() - start

print("\n==============================")
print("BENCHMARK")
print("==============================")
print(f"200 steps: {elapsed / 60:.2f} minutes")
print(f"Average: {elapsed / 200:.3f} sec/step")

estimated_hours = (elapsed / 200) * 581008 / 3600

print(f"Estimated full 581,008 steps: {estimated_hours:.2f} GPU hours")
print(f"30-hour budget remaining margin: {30 - estimated_hours:.2f} hours")


In [ ]:
from pathlib import Path
import shutil

src = Path(
    "/kaggle/input/datasets/thothanh28/"
    "checkpoint58100/checkpoint-58100"
)

dst = Path(
    "/kaggle/working/MultiMend/models/"
    "multimend-codet5-small/checkpoint-58100"
)

if not src.exists():
    raise FileNotFoundError(f"Checkpoint không tồn tại: {src}")

dst.parent.mkdir(parents=True, exist_ok=True)

if dst.exists():
    shutil.rmtree(dst)

shutil.copytree(src, dst)

print("Restored checkpoint:")
print(dst)
print()
print("Files:")

for p in sorted(dst.iterdir()):
    print(" -", p.name)

In [ ]:
from pathlib import Path
import shutil

src = Path(
    "/kaggle/input/datasets/honglcl/"
    "checkpoint-116200/checkpoint-116200"
)

dst = Path(
    "/kaggle/working/MultiMend/models/"
    "multimend-codet5-small/checkpoint-116200"
)

if not src.exists():
    raise FileNotFoundError(f"Checkpoint không tồn tại: {src}")

dst.parent.mkdir(parents=True, exist_ok=True)

if dst.exists():
    shutil.rmtree(dst)

shutil.copytree(src, dst)

print("Restored checkpoint:")
print(dst)
print()
print("Files:")

for p in sorted(dst.iterdir()):
    print(" -", p.name)


In [ ]:
from pathlib import Path
import shutil

src = Path(
    "/kaggle/input/datasets/honglcl/"
    "checkpoint174300/checkpoint-174300"
)

dst = Path(
    "/kaggle/working/MultiMend/models/"
    "multimend-codet5-small/checkpoint-174300"
)

if not src.exists():
    raise FileNotFoundError(f"Checkpoint không tồn tại: {src}")

dst.parent.mkdir(parents=True, exist_ok=True)

if dst.exists():
    shutil.rmtree(dst)

shutil.copytree(src, dst)

print("Restored checkpoint:")
print(dst)
print()
print("Files:")

for p in sorted(dst.iterdir()):
    print(" -", p.name)


In [ ]:
from pathlib import Path
import shutil

src = Path(
    "/kaggle/input/datasets/minhhanh123/"
    "checkpoint-232400/checkpoint-232400"
)

dst = Path(
    "/kaggle/working/MultiMend/models/"
    "multimend-codet5-small/checkpoint-232400"
)

if not src.exists():
    raise FileNotFoundError(f"Checkpoint không tồn tại: {src}")

dst.parent.mkdir(parents=True, exist_ok=True)

if dst.exists():
    shutil.rmtree(dst)

shutil.copytree(src, dst)

print("Restored checkpoint:")
print(dst)
print()
print("Files:")

for p in sorted(dst.iterdir()):
    print(" -", p.name)

In [ ]:
from pathlib import Path
import shutil

src = Path(
    "/kaggle/input/datasets/tramtvt/"
    "checkpoint348600/checkpoint-348600"
)

dst = Path(
    "/kaggle/working/MultiMend/models/"
    "multimend-codet5-small/checkpoint-348600"
)

if not src.exists():
    raise FileNotFoundError(f"Checkpoint không tồn tại: {src}")

dst.parent.mkdir(parents=True, exist_ok=True)

if dst.exists():
    shutil.rmtree(dst)

shutil.copytree(src, dst)

print("Restored checkpoint:")
print(dst)
print()
print("Files:")

for p in sorted(dst.iterdir()):
    print(" -", p.name)


In [ ]:
from pathlib import Path
import shutil

src = Path(
    "/kaggle/input/datasets/tramtvt/checkpoint406700/"
    "checkpoint-406700"
)

dst = Path(
    "/kaggle/working/MultiMend/models/"
    "multimend-codet5-small/checkpoint-406700"
)

if not src.exists():
    raise FileNotFoundError(f"Checkpoint không tồn tại: {src}")

dst.parent.mkdir(parents=True, exist_ok=True)

if dst.exists():
    shutil.rmtree(dst)

shutil.copytree(src, dst)

print("Restored checkpoint:")
print(dst)
print()
print("Files:")

for p in sorted(dst.iterdir()):
    print(" -", p.name)


In [ ]:
from pathlib import Path
import shutil

src = Path(
    "/kaggle/input/datasets/nguynhongbchtrm/"
    "checkpoint464800/checkpoint-464800"
)

dst = Path(
    "/kaggle/working/MultiMend/models/"
    "multimend-codet5-small/checkpoint-464800"
)

if not src.exists():
    raise FileNotFoundError(f"Checkpoint không tồn tại: {src}")

dst.parent.mkdir(parents=True, exist_ok=True)

if dst.exists():
    shutil.rmtree(dst)

shutil.copytree(src, dst)

print("Restored checkpoint:")
print(dst)
print()
print("Files:")

for p in sorted(dst.iterdir()):
    print(" -", p.name)


In [ ]:
from pathlib import Path
import shutil

src = Path(
    "/kaggle/input/datasets/nguynhongbchtrm/"
    "checkpoint522900/checkpoint-522900"
)

dst = Path(
    "/kaggle/working/MultiMend/models/"
    "multimend-codet5-small/checkpoint-522900"
)

if not src.exists():
    raise FileNotFoundError(f"Checkpoint không tồn tại: {src}")

dst.parent.mkdir(parents=True, exist_ok=True)

if dst.exists():
    shutil.rmtree(dst)

shutil.copytree(src, dst)

print("Restored checkpoint:")
print(dst)
print()
print("Files:")

for p in sorted(dst.iterdir()):
    print(" -", p.name)


In [ ]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    for d in dirs:
        if "522900" in d:
            print("FOUND:", os.path.join(root, d))


In [ ]:
import os

print("WORLD_SIZE:", os.environ.get("WORLD_SIZE"))
print("LOCAL_RANK:", os.environ.get("LOCAL_RANK"))
print("RANK:", os.environ.get("RANK"))


mục tiêu giữ nguyên training config của MultiMend và chỉ thêm cơ chế auto-resume, dùng cell này. Nó sẽ tìm checkpoint-* mới nhất trong models/multimend-codet5-small/; nếu chưa có thì train từ đầu, nếu có thì resume từ checkpoint đó.

In [ ]:
# Cell 12 — Official MultiMend training with automatic checkpoint resume
#
# IMPORTANT:
# - Keeps the official MultiMend training configuration unchanged.
# - Only adds automatic checkpoint detection/resume.
# - Run this cell again in a new Kaggle session after restoring the
#   checkpoint directory.
#
# Expected checkpoint layout:
# /kaggle/working/MultiMend/models/multimend-codet5-small/
# ├── checkpoint-58100/
# ├── checkpoint-116200/
# ├── ...
#
# Do NOT delete optimizer.pt, scheduler.pt, trainer_state.json, etc.
# from the checkpoint if you want a proper training resume.

from pathlib import Path
import re
import torch

from transformers import (
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    DataCollatorForSeq2Seq,
    GenerationConfig,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
)

# ------------------------------------------------------------------
# 1. Paths
# ------------------------------------------------------------------

REPO = Path("/kaggle/working/MultiMend")
MODELS_ROOT = REPO / "models"
OUTPUT_DIR = MODELS_ROOT / "multimend-codet5-small"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Repository:", REPO)
print("Training output:", OUTPUT_DIR)
print("CUDA:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA:", torch.version.cuda)


# ------------------------------------------------------------------
# 2. Auto-detect latest valid checkpoint
# ------------------------------------------------------------------

def get_latest_checkpoint(output_dir: Path):
    checkpoints = []

    for p in output_dir.glob("checkpoint-*"):
        if not p.is_dir():
            continue

        match = re.fullmatch(r"checkpoint-(\d+)", p.name)
        if match:
            step = int(match.group(1))
            checkpoints.append((step, p))

    if not checkpoints:
        return None

    checkpoints.sort(key=lambda x: x[0])

    return checkpoints[-1][1]


latest_checkpoint = get_latest_checkpoint(OUTPUT_DIR)

if latest_checkpoint is None:
    print("\nNo existing checkpoint found.")
    print("Training will start from Salesforce/codet5-small.")
else:
    print("\nLatest checkpoint found:")
    print(" ", latest_checkpoint)

    # Useful sanity check
    required_files = [
        "trainer_state.json",
        "optimizer.pt",
        "scheduler.pt",
    ]

    missing = [
        f for f in required_files
        if not (latest_checkpoint / f).exists()
    ]

    if missing:
        raise RuntimeError(
            "Checkpoint appears incomplete.\n"
            f"Checkpoint: {latest_checkpoint}\n"
            f"Missing files: {missing}\n\n"
            "Do not resume from an incomplete checkpoint."
        )

    print("Checkpoint training state looks complete.")


# ------------------------------------------------------------------
# 3. Official base model
# ------------------------------------------------------------------

MODEL_CHECKPOINT = "Salesforce/codet5-small"

tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_CHECKPOINT)


# ------------------------------------------------------------------
# 4. IMPORTANT:
#    Reuse the tokenized/filtered training dataset from Cell 8/earlier.
#
#    This variable should already exist in the notebook:
#
#        concatenated_dataset
#
#    We deliberately do NOT recreate the 2.3M-example preprocessing here.
# ------------------------------------------------------------------

if "concatenated_dataset" not in globals():
    raise RuntimeError(
        "concatenated_dataset is not available.\n"
        "Run the official dataset preprocessing cell first."
    )

print("\nTraining dataset:")
print(concatenated_dataset)


# ------------------------------------------------------------------
# 5. Official MultiMend generation configuration
# ------------------------------------------------------------------

max_input_length = 512
max_target_length = 256

generation_config = GenerationConfig(
    max_length=max_target_length,
    min_length=0,
    early_stopping=True,
    num_beams=5,
)


# ------------------------------------------------------------------
# 6. Official MultiMend training configuration
# ------------------------------------------------------------------

train_batch_size = 8
eval_batch_size = 16

epochs = 2
lr = 1e-4

checkpoints_each_epoch = 5

epoch_steps = (
    len(concatenated_dataset) // train_batch_size
    + int(len(concatenated_dataset) % train_batch_size != 0)
)

train_steps = epoch_steps * epochs

save_steps = epoch_steps // checkpoints_each_epoch

print("\n==============================")
print("OFFICIAL TRAINING CONFIG")
print("==============================")
print("Dataset size:", len(concatenated_dataset))
print("Train batch size:", train_batch_size)
print("Epoch steps:", epoch_steps)
print("Total training steps:", train_steps)
print("Save steps:", save_steps)
print("Epochs:", epochs)
print("Learning rate:", lr)
print("FP16:", True)


# ------------------------------------------------------------------
# 7. Trainer
# ------------------------------------------------------------------

args = Seq2SeqTrainingArguments(
    output_dir=str(OUTPUT_DIR),

    learning_rate=lr,

    per_device_train_batch_size=train_batch_size,
    per_device_eval_batch_size=eval_batch_size,

    save_total_limit=epochs * checkpoints_each_epoch,

    max_steps=train_steps,

    save_steps=save_steps,

    predict_with_generate=True,

    fp16=True,

    lr_scheduler_type="constant",

    generation_config=generation_config,

    report_to="tensorboard",
)

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=concatenated_dataset,
    data_collator=data_collator,
    tokenizer=tokenizer,
)


# ------------------------------------------------------------------
# 8. AUTO RESUME
# ------------------------------------------------------------------

print("\n==============================")
print("TRAINING")
print("==============================")

if latest_checkpoint is None:

    print("Starting training from scratch...")
    
    trainer.train(
        resume_from_checkpoint=False
    )

else:

    print(f"Resuming from: {latest_checkpoint}")

    trainer.train(
        resume_from_checkpoint=str(latest_checkpoint)
    )


# ------------------------------------------------------------------
# 9. Final status
# ------------------------------------------------------------------

print("\n==============================")
print("TRAINING SESSION FINISHED")
print("==============================")

latest_checkpoint = get_latest_checkpoint(OUTPUT_DIR)

if latest_checkpoint:
    print("Latest checkpoint:")
    print(latest_checkpoint)

    state_file = latest_checkpoint / "trainer_state.json"

    if state_file.exists():
        import json

        with open(state_file, "r") as f:
            state = json.load(f)

        print("Global step:", state.get("global_step"))
        print("Epoch:", state.get("epoch"))

print("\nAll checkpoint directories:")

for p in sorted(
    OUTPUT_DIR.glob("checkpoint-*"),
    key=lambda x: int(x.name.split("-")[-1])
):
    print(" ", p.name)


In [ ]:
from pathlib import Path

ckpt = Path(
    "/kaggle/working/MultiMend/"
    "models/multimend-codet5-small/checkpoint-581008"
)

print("Exists:", ckpt.exists())

if ckpt.exists():
    print("Files:")
    for p in ckpt.iterdir():
        print(" -", p.name)


In [ ]:
import shutil
from pathlib import Path

ckpt = Path(
    "/kaggle/working/MultiMend/"
    "models/multimend-codet5-small/checkpoint-581008"
)

zip_path = Path("/kaggle/working/checkpoint-581008")

shutil.make_archive(
    str(zip_path),
    "zip",
    root_dir=ckpt.parent,
    base_dir=ckpt.name
)

final_zip = Path("/kaggle/working/checkpoint-581008.zip")

print("Created:", final_zip)
print("Size:", round(final_zip.stat().st_size / 1024**3, 2), "GB")


In [ ]:
!kaggle config view


In [ ]:
from pathlib import Path

zip_path = Path("/kaggle/working/checkpoint-174300.zip")

print("Exists:", zip_path.exists())

if zip_path.exists():
    print("Path:", zip_path)
    print("Size:", round(zip_path.stat().st_size / 1024**2, 2), "MB")
